# Unified Stellar Catalog & RV Homogenisation — Publication Figures

**Part I — Catalog Visualization** (Figures 1–6)
1. Sky Grid (4×4 Mollweide, all surveys + PRISTINE + CAT)
2. Extinction Sky Map
3. HR Diagram (gray unfiltered + coloured filtered + external contours + Teff axis)
4. Kiel Diagram (single panel: gray unfiltered + coloured filtered)
5. RV vs Distance Phase-Space (main catalog density + own contours, no cluster members)
6. Error Analysis (N vs Distance + Median Error vs Distance)

**Part II — RV Normalization** (Figures 7–17 + Supplementary)

**Workflow:** Run cells 0–3 sequentially (imports, paths, helpers, data loading). Then run any figure cell independently.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Imports & Publication Style
# ═══════════════════════════════════════════════════════════════
import os, sys, re, warnings, pickle, glob
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, Normalize, LinearSegmentedColormap
from matplotlib.colorbar import ColorbarBase
from matplotlib.ticker import AutoMinorLocator, NullLocator
from matplotlib import patches
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
from scipy.stats import norm, gaussian_kde, median_abs_deviation
from scipy.ndimage import gaussian_filter
warnings.filterwarnings('ignore')
%matplotlib inline

try:
    import healpy as hp
    HEALPY_OK = True
    print('✓ healpy')
except ImportError:
    HEALPY_OK = False
    print('✗ healpy not found — sky maps disabled')

try:
    from astropy.io import fits as afits
    print('✓ astropy')
except ImportError:
    print('✗ astropy not found')

# ── Publication-quality rcParams ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family':         'serif',
    'font.serif':          ['DejaVu Serif', 'Times', 'Palatino'],
    'font.size':           16,
    'axes.labelsize':      24,
    'axes.titlesize':      22,
    'axes.titleweight':    'bold',
    'xtick.labelsize':     18,
    'ytick.labelsize':     18,
    'legend.fontsize':     12,
    'xtick.major.size':    10,  'xtick.major.width':  2.0,
    'ytick.major.size':    10,  'ytick.major.width':  2.0,
    'xtick.minor.size':    5,   'xtick.minor.width':  1.2,
    'ytick.minor.size':    5,   'ytick.minor.width':  1.2,
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.top':           True,
    'ytick.right':         True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'axes.linewidth':      2.0,
    'axes.facecolor':      'white',
    'figure.facecolor':    'white',
    'savefig.facecolor':   'white',
    'figure.dpi':          150,
    'savefig.dpi':         300,
    'savefig.bbox':        'tight',
    'lines.linewidth':     1.8,
    'grid.alpha':          0.3,
    'grid.linestyle':      '--',
})
print('✓ Publication rcParams applied')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Paths & Column Definitions
# ═══════════════════════════════════════════════════════════════
WORK      = '/user/sutirtha/BallTree_Xmatch'
NPZ_V5    = os.path.join(WORK, 'stellar_data_v5.npz')
NPZ_PATCH = os.path.join(WORK, 'stellar_data_v5_patched.npz')
RV_DIR    = os.path.join(WORK, 'rv_results')
CACHE_NPZ = os.path.join(RV_DIR, 'master_cache.npz')
CACHE_MEM = os.path.join(RV_DIR, 'member_cache.pkl')
DAT_DIR   = os.path.join(WORK,   'astro_data/A95_cds')
OUT_DIR   = os.path.join(WORK,   'notebook_figs')
os.makedirs(OUT_DIR, exist_ok=True)

FITS_PATTERN = os.path.join(WORK, 'Entire_catalogue_chunk*.fits')
FITS_FILES   = sorted(glob.glob(FITS_PATTERN))

# ── Homo_rv paths ─────────────────────────────────────────────────────────────
CKPT_DIR = Path(os.path.join(WORK, 'rv_norm_ckpt_v9'))
HOMO_OUT = Path(os.path.join(WORK, 'rv_norm_output_v9'))

# ── External survey .dat files ────────────────────────────────────────────────
DAT_FILES = {
    'APOGEE':     os.path.join(DAT_DIR, 'apogee.dat'),
    'GALAH':      os.path.join(DAT_DIR, 'galah.dat'),
    'GES':        os.path.join(DAT_DIR, 'ges.dat'),
    'RAVE':       os.path.join(DAT_DIR, 'rave.dat'),
    'LAMOST_ext': os.path.join(DAT_DIR, 'lamost.dat'),
}

# ── Member CSVs ───────────────────────────────────────────────────────────────
MEMBER_FILES = {
    'GC':  (os.path.join(WORK, 'GC_members_with_RV.csv'),  'ra',    'dec'),
    'OC':  (os.path.join(WORK, 'OC_members.csv'),           'RAdeg', 'DEdeg'),
    'SGR': (os.path.join(WORK, 'Sgr_stream_members.csv'),   'ra',    'dec'),
    'DWG': (os.path.join(WORK, 'DWG_members.csv'),          'ra_x',  'dec_x'),
}

NSIDE      = 64
CHUNK_SIZE = 2_000_000

# ── FITS column names ─────────────────────────────────────────────────────────
COL_BPRP_OBS  = 'BP-RP'
COL_EBPRP     = 'E(BP-RP)'
COL_GMAG      = 'Gmag'
COL_AG        = 'AG'
COL_DIST      = 'distance_final'
COL_RA        = 'RAdeg'
COL_DEC       = 'DEdeg'
COL_TEFF      = 'Teff'
COL_LOGG      = 'logg'
COL_FEH       = 'feh'
COL_CODE      = 'Code_combined'
COL_SURVEY    = 'Survey_combined'
COL_WA        = 'Weighted_Avg_final'
COL_WA_ERR    = 'Weighted_Avg_err_final'
COL_ZP        = 'ZP_final'
COL_ZP_ERR    = 'ZP_err_final'
COL_RV        = 'RV_final'
COL_RV_ERR    = 'RV_err_final'
COL_NMEAS     = 'n_measurements'
COL_RVSN      = 'RVS/N'
COL_RUWE      = 'RUWE'

# ── Homo_rv survey colours ────────────────────────────────────────────────────
COLORS = {
    'GAIA':   '#1f77b4', 'APOGEE': '#2ca02c', 'LAMOST': '#d62728',
    'RAVE':   '#ff7f0e', 'GALAH':  '#9467bd', 'GES':    '#8c564b',
    'DESI':   '#e377c2', 'SDSS':   '#17becf',
}
def _c(s): return COLORS.get(s, '#555555')

SAVE_PNG = True
SAVE_PDF = True

def savefig(fig, name, out_dir=None):
    d = Path(out_dir) if out_dir else Path(OUT_DIR)
    if SAVE_PNG: fig.savefig(d / f'{name}.png')
    if SAVE_PDF: fig.savefig(d / f'{name}.pdf')
    print(f"  ✓ Saved → {d / name}")

print(f'FITS files found: {len(FITS_FILES)}')
for k, v in {'stellar_data_v5.npz': NPZ_V5,
             'patched NPZ': NPZ_PATCH,
             'master_cache.npz': CACHE_NPZ,
             'member_cache.pkl': CACHE_MEM}.items():
    print(f'  {"✓" if os.path.exists(v) else "✗ MISSING":10s} {k}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Shared Helpers
# ═══════════════════════════════════════════════════════════════
def ra_dec_to_galactic(ra, dec):
    ra_r = np.radians(ra); dec_r = np.radians(dec)
    x = np.cos(dec_r)*np.cos(ra_r)
    y = np.cos(dec_r)*np.sin(ra_r)
    z = np.sin(dec_r)
    R = np.array([[-0.054875539390,-0.873437104725,-0.483834991775],
                  [ 0.494109453633,-0.444829594298, 0.746982248696],
                  [-0.867666135681,-0.198076389622, 0.455983794523]])
    xyz = R @ np.array([x, y, z])
    l = np.degrees(np.arctan2(xyz[1], xyz[0]))
    b = np.degrees(np.arcsin(np.clip(xyz[2], -1, 1)))
    return np.mod(l, 360.0), b

def make_healpix_map(ra, dec, nside=NSIDE):
    if not HEALPY_OK: return None
    ok = np.isfinite(ra) & np.isfinite(dec)
    if not np.any(ok): return np.zeros(hp.nside2npix(nside))
    l, b  = ra_dec_to_galactic(ra[ok], dec[ok])
    theta = np.radians(90.0 - b); phi = np.radians(l)
    npix  = hp.nside2npix(nside)
    cnt   = np.zeros(npix, dtype=np.float64)
    pix   = hp.ang2pix(nside, theta, phi)
    np.add.at(cnt, pix, 1)
    return cnt

def mollweide_grid(nside=NSIDE, n_lon=1440, n_lat=720):
    l_g = np.linspace(-180,180,n_lon); b_g = np.linspace(-90,90,n_lat)
    L, B = np.meshgrid(l_g, b_g)
    theta = np.radians(90.0-B.ravel())
    phi   = np.radians(np.mod(L.ravel(),360))
    pix   = hp.ang2pix(nside, theta, phi)
    return L, B, -np.radians(L), np.radians(B), pix

def radec_to_moll(ra_deg, dec_deg):
    l, b = ra_dec_to_galactic(ra_deg, dec_deg)
    x = -np.radians(l)
    x = np.where(x >  np.pi, x - 2*np.pi, x)
    x = np.where(x < -np.pi, x + 2*np.pi, x)
    y = np.radians(b)
    return x, y

def style_mollweide_white(ax, title, n_stars=0):
    """Publication-quality Mollweide frame with WHITE background."""
    ax.set_facecolor('white')
    ax.grid(True, alpha=0.25, ls='-', lw=0.5, color='gray')
    lt = np.array([150,120,90,60,30,0,-30,-60,-90,-120,-150])
    ax.set_xticks(np.radians(lt))
    ax.set_xticklabels([f'{int(np.mod(-ld,360))}°' for ld in lt],
                        fontsize=9, color='black')
    ax.set_yticks(np.radians([-60,-30,0,30,60]))
    ax.set_yticklabels([f'{ld}°' for ld in [-60,-30,0,30,60]],
                        fontsize=9, color='black')
    ns = f'  N={n_stars:,}' if n_stars > 0 else ''
    ax.set_title(f'{title}{ns}', fontsize=13, fontweight='bold',
                 color='black', pad=5)
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')

def add_density_contours(ax, hist2d, x_edges, y_edges,
                         n_levels=5, color='white', lw=0.9, sigma=3.0):
    h = gaussian_filter(hist2d.copy().astype(float), sigma=sigma)
    nz = h[h>0]
    if len(nz)<10: return
    levs = np.unique(np.percentile(nz, np.linspace(20,95,n_levels)))
    if len(levs)<2: return
    xc = (x_edges[:-1]+x_edges[1:])/2
    yc = (y_edges[:-1]+y_edges[1:])/2
    ax.contour(xc, yc, h, levels=levs,
               colors=color, linewidths=lw, alpha=0.75, zorder=12)

def safe_vmax(arr, pct=99.5):
    fin = arr[np.isfinite(arr) & (arr>0)]
    if len(fin)==0: return None
    v = float(np.nanpercentile(fin, pct))
    if not np.isfinite(v) or v<1: v = float(np.nanmax(fin))
    return max(v, 2.0)

def make_magma_cmap():
    cols = plt.cm.get_cmap('magma')(np.linspace(0.13,0.97,256))
    cm = LinearSegmentedColormap.from_list('magma_nb', cols, 256)
    cm.set_under('white'); cm.set_bad('white')
    return cm

# ── Homo_rv helpers ───────────────────────────────────────────────────────────
def bin_stat(x, y, n_bins=40, p_lo=1, p_hi=99, min_count=10):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < min_count*2: return np.array([]), np.array([]), np.array([])
    x, y = x[ok], y[ok]
    lo, hi = np.percentile(x, p_lo), np.percentile(x, p_hi)
    be = np.linspace(lo, hi, n_bins+1)
    bc, meds, mads = [], [], []
    for j in range(n_bins):
        m = (x >= be[j]) & (x < be[j+1])
        if m.sum() < min_count: continue
        yy = y[m]; med = float(np.median(yy))
        mads.append(float(np.median(np.abs(yy-med))))
        bc.append(0.5*(be[j]+be[j+1])); meds.append(med)
    return np.array(bc), np.array(meds), np.array(mads)

def bin_stat_counts(x, y, n_bins=30, p_lo=1, p_hi=99, min_count=10):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < min_count*2: return np.array([]),np.array([]),np.array([]),np.array([])
    x, y = x[ok], y[ok]
    lo, hi = np.percentile(x, p_lo), np.percentile(x, p_hi)
    be = np.linspace(lo, hi, n_bins+1)
    bc, meds, mads, ns = [], [], [], []
    for j in range(n_bins):
        m = (x >= be[j]) & (x < be[j+1])
        if m.sum() < min_count: continue
        yy = y[m]; med = float(np.median(yy))
        bc.append(0.5*(be[j]+be[j+1])); meds.append(med)
        mads.append(float(np.median(np.abs(yy-med)))); ns.append(int(m.sum()))
    return np.array(bc), np.array(meds), np.array(mads), np.array(ns)

def _poly_err(coeffs, cov, x_val):
    if cov is None: return 0.0
    try:
        n = len(coeffs)-1
        grad = np.array([float(x_val)**(n-i) for i in range(n+1)])
        cov  = np.asarray(cov, dtype=np.float64)
        var  = float(grad @ cov @ grad)
        if not np.isfinite(var) or var < 0: return np.nan
        r = float(np.sqrt(var))
        return r if r <= 5.0 else np.nan
    except: return 0.0

def add_panel_label(ax, label, x=0.02, y=0.96, fontsize=13, bold=True):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=fontsize,
            fontweight='bold' if bold else 'normal',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))

print('✓ All helpers defined')


## Data Loading
Load NPZ, master cache, member cache, external surveys, and Homo_rv checkpoints/CSVs.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3a — Load stellar_data_v5.npz
# ═══════════════════════════════════════════════════════════════
npz_to_load = NPZ_PATCH if os.path.exists(NPZ_PATCH) else NPZ_V5
print(f'Loading: {npz_to_load}')
raw = np.load(npz_to_load, allow_pickle=False)
D = {k: raw[k] for k in raw.files}

bprp_edges = D['_bprp_edges']
mg_edges   = D['_mg_edges']
teff_edges = D['_teff_edges']
logg_edges = D['_logg_edges']

hr_unfilt   = D['hr_full_unfilt'].astype(float)
hr_filt     = D['hr_full_filt'].astype(float)
kiel_unfilt = D['kiel_full_unfilt'].astype(float)
kiel_filt   = D['kiel_full_filt'].astype(float)
teff_mean   = D.get('teff_bprp_mean')
feh_mean    = D.get('kiel_feh_mean')

n_total = int(D.get('_n_rows',     np.array([0]))[0])
n_filt  = int(D.get('_n_filtered', np.array([0]))[0])

print(f'  Total rows  : {n_total:,}')
print(f'  Filtered rows: {n_filt:,}')
for k in ['hr_full_filt','hr_full_unfilt','kiel_full_filt',
          'ext_ebprp_mean','ext_ag_mean','teff_bprp_mean',
          'sky_full_filt','sky_SOS','sky_Gaia125M','sky_Gaia33M',
          'sky_LAMOST','sky_DESI','sky_SDSS']:
    if k in D:
        arr = D[k].astype(float)
        nz  = int(np.sum(arr>0))
        status = '✓' if nz>0 else '✗ EMPTY'
        print(f'  {status:10s} {k:30s} nonzero={nz:,}')
    else:
        print(f'  {"✗ MISSING":10s} {k}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3b — Recompute from FITS (skip if patched NPZ exists)
# ═══════════════════════════════════════════════════════════════
if os.path.exists(NPZ_PATCH):
    print(f'Patched NPZ already exists: {NPZ_PATCH}')
    print('Skipping recompute — delete the patched NPZ to force rerun')
else:
    print(f'Recomputing from {len(FITS_FILES)} FITS files...')
    print('This will take ~15-30 minutes\n')

    npix = hp.nside2npix(NSIDE)
    hr_u_new   = np.zeros((600, 600), dtype=np.float64)
    hr_f_new   = np.zeros((600, 600), dtype=np.float64)
    ebprp_sum  = np.zeros(npix, dtype=np.float64)
    ag_sum     = np.zeros(npix, dtype=np.float64)
    ext_cnt    = np.zeros(npix, dtype=np.float64)
    teff_bsum  = np.zeros(600, dtype=np.float64)
    teff_bcnt  = np.zeros(600, dtype=np.float64)
    sky_SOS_new  = np.zeros(npix, dtype=np.float64)
    sky_125m_new = np.zeros(npix, dtype=np.float64)
    sky_33m_new  = np.zeros(npix, dtype=np.float64)

    SRV_PATS = {
        'LAMOST':   re.compile(r'lamost',           re.IGNORECASE),
        'SDSS':     re.compile(r'sdss|boss|segue',  re.IGNORECASE),
        'DESI':     re.compile(r'desi',             re.IGNORECASE),
        'PRISTINE': re.compile(r'pristine',         re.IGNORECASE),
        'CAT':      re.compile(r'cat',              re.IGNORECASE),
    }
    sky_survey_new = {k: np.zeros(npix, dtype=np.float64) for k in SRV_PATS}

    total_hr = 0; total_rows = 0

    for fi, fp in enumerate(FITS_FILES):
        print(f'[{fi+1}/{len(FITS_FILES)}] {os.path.basename(fp)}')
        with afits.open(fp, memmap=True) as hdul:
            dh = hdul[1]
            nrows_file = dh.header['NAXIS2']
            for start in range(0, nrows_file, CHUNK_SIZE):
                end   = min(start + CHUNK_SIZE, nrows_file)
                chunk = dh.data[start:end]
                n     = end - start
                total_rows += n

                def gc(name):
                    if name in chunk.dtype.names:
                        return np.array(chunk[name], dtype=np.float64)
                    return np.full(n, np.nan)

                bprp_obs = gc(COL_BPRP_OBS)
                ebprp    = gc(COL_EBPRP)
                gmag     = gc(COL_GMAG)
                ag       = gc(COL_AG)
                dist     = gc(COL_DIST)
                ra       = gc(COL_RA)
                dec      = gc(COL_DEC)
                teff     = gc(COL_TEFF)
                ruwe     = gc(COL_RUWE)

                bprp0 = bprp_obs - ebprp
                with np.errstate(invalid='ignore', divide='ignore'):
                    dist_pc = dist * 1000.0
                    dm      = 5.0 * np.log10(dist_pc) - 5.0
                    mg_abs  = gmag - dm - np.where(np.isfinite(ag), ag, 0.0)

                mask_f = (
                    (np.isfinite(ruwe) & (ruwe <= 1.4)) |
                    (~np.isfinite(ruwe))
                )
                if not np.any(np.isfinite(ruwe)):
                    mask_f = np.ones(n, dtype=bool)

                ok_hr  = np.isfinite(bprp0) & np.isfinite(mg_abs)
                ok_hrf = ok_hr & mask_f
                if np.any(ok_hr):
                    h, _, _ = np.histogram2d(bprp0[ok_hr], mg_abs[ok_hr],
                                             bins=[bprp_edges, mg_edges])
                    hr_u_new += h.T
                    total_hr += int(np.sum(ok_hr))
                if np.any(ok_hrf):
                    h, _, _ = np.histogram2d(bprp0[ok_hrf], mg_abs[ok_hrf],
                                             bins=[bprp_edges, mg_edges])
                    hr_f_new += h.T
                    ok_t = ok_hrf & np.isfinite(teff)
                    if np.any(ok_t):
                        idx_b = np.clip(
                            np.searchsorted(bprp_edges, bprp0[ok_t])-1, 0, 599)
                        np.add.at(teff_bsum, idx_b, teff[ok_t])
                        np.add.at(teff_bcnt, idx_b, 1)

                ok_pos = np.isfinite(ra) & np.isfinite(dec)
                if np.any(ok_pos):
                    l, b = ra_dec_to_galactic(ra[ok_pos], dec[ok_pos])
                    theta = np.radians(90.0-b); phi = np.radians(l)
                    pix   = hp.ang2pix(NSIDE, theta, phi)
                    ok_e  = np.isfinite(ebprp[ok_pos])
                    if np.any(ok_e):
                        np.add.at(ebprp_sum, pix[ok_e], ebprp[ok_pos][ok_e])
                        np.add.at(ext_cnt,   pix[ok_e], 1)
                    ok_ag = np.isfinite(ag[ok_pos])
                    if np.any(ok_ag):
                        np.add.at(ag_sum, pix[ok_ag], ag[ok_pos][ok_ag])

                if COL_CODE in chunk.dtype.names and np.any(ok_pos):
                    codes     = np.array(chunk[COL_CODE], dtype=str)
                    codes_pos = codes[ok_pos]
                    pix_all   = pix
                    mask_sos = np.array(['DSOS' in c for c in codes_pos])
                    mask_125 = np.array(['D125' in c for c in codes_pos])
                    mask_33  = np.array(['D33M' in c for c in codes_pos])
                    if np.any(mask_sos): np.add.at(sky_SOS_new,  pix_all[mask_sos], 1)
                    if np.any(mask_125): np.add.at(sky_125m_new, pix_all[mask_125], 1)
                    if np.any(mask_33):  np.add.at(sky_33m_new,  pix_all[mask_33],  1)

                if COL_SURVEY in chunk.dtype.names and np.any(ok_pos):
                    srvs_pos = np.array(chunk[COL_SURVEY], dtype=str)[ok_pos]
                    for srv_key, pat in SRV_PATS.items():
                        srv_mask = np.array([bool(pat.search(s)) for s in srvs_pos])
                        if np.any(srv_mask):
                            np.add.at(sky_survey_new[srv_key], pix_all[srv_mask], 1)

                print(f'  chunk {start//CHUNK_SIZE+1}: rows={n:,} '
                      f'HR_ok={int(np.sum(ok_hr)):,}', flush=True)

    print(f'\nDone. Total rows={total_rows:,}  HR stars={total_hr:,}')

    with np.errstate(invalid='ignore', divide='ignore'):
        teff_mean_new = np.where(teff_bcnt>0, teff_bsum/teff_bcnt, np.nan)
        ext_ebprp_new = np.where(ext_cnt>0,   ebprp_sum/ext_cnt,   np.nan)
        ext_ag_new    = np.where(ext_cnt>0,   ag_sum/ext_cnt,      np.nan)

    D['hr_full_unfilt']  = hr_u_new.astype(np.float32)
    D['hr_full_filt']    = hr_f_new.astype(np.float32)
    D['teff_bprp_mean']  = teff_mean_new.astype(np.float32)
    D['ext_ebprp_mean']  = ext_ebprp_new.astype(np.float32)
    D['ext_ag_mean']     = ext_ag_new.astype(np.float32)
    D['sky_SOS']         = sky_SOS_new.astype(np.float32)
    D['sky_Gaia125M']    = sky_125m_new.astype(np.float32)
    D['sky_Gaia33M']     = sky_33m_new.astype(np.float32)
    key_map = {'LAMOST':'sky_LAMOST','SDSS':'sky_SDSS','DESI':'sky_DESI',
               'PRISTINE':'sky_PRISTINE','CAT':'sky_CAT'}
    for k, arr in sky_survey_new.items():
        if k in key_map and np.any(arr>0):
            D[key_map[k]] = arr.astype(np.float32)

    hr_unfilt  = D['hr_full_unfilt'].astype(float)
    hr_filt    = D['hr_full_filt'].astype(float)
    teff_mean  = D['teff_bprp_mean']

    np.savez_compressed(NPZ_PATCH, **D)
    sz = os.path.getsize(NPZ_PATCH)/1e6
    print(f'\n✓ Patched NPZ saved: {NPZ_PATCH}  ({sz:.1f} MB)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3c — Load master_cache + member_cache + external surveys
# ═══════════════════════════════════════════════════════════════
print('Loading master_cache.npz ...')
_m = np.load(CACHE_NPZ, allow_pickle=False)
master = {k: _m[k] for k in _m.files}
if 'n_measurements' not in master and 'RVNper' in master:
    master['n_measurements'] = master.pop('RVNper')
print(f'  Rows: {len(master[COL_DIST]):,}')

print('Loading member_cache.pkl ...')
with open(CACHE_MEM, 'rb') as f:
    members = pickle.load(f)
for k, df in members.items():
    print(f'  {k}: {len(df):,} rows')

# ── External .dat surveys ─────────────────────────────────────────────────────
DAT_COLS = ['SoS','Name','GaiaDR2','RAdeg','DEdeg','RVcor','e_RVcor',
            's_RVcor','N','flagXM','flagRV','flagBinary',
            'Teff','e_Teff','logg','e_logg','FeH','e_FeH']

surveys_ext = {}
print('\nLoading external .dat surveys ...')
for name, path in DAT_FILES.items():
    if not os.path.exists(path):
        print(f'  ✗ MISSING: {name}'); continue
    try:
        df = pd.read_csv(path, sep=r'\s+', comment='#',
                         header=None, names=DAT_COLS,
                         na_values=['-999','---','NaN','','nan'])
    except Exception as e:
        print(f'  ✗ {name}: {e}'); continue
    df.columns = [c.lower() for c in df.columns]
    for col in ['radeg','dedeg','rvcor','teff','logg','feh']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    surveys_ext[name] = df
    ok_sky  = df['radeg'].notna() & df['dedeg'].notna()
    ok_kiel = df['teff'].notna()  & df['logg'].notna()
    print(f'  ✓ {name:12s}: {len(df):>8,} rows | '
          f'sky={ok_sky.sum():>8,} | kiel={ok_kiel.sum():>8,}')

# HEALPix sky maps for external surveys
print('\nBuilding sky maps for external surveys ...')
sky_ext = {}
for name, df in surveys_ext.items():
    ok  = df['radeg'].notna() & df['dedeg'].notna()
    ra  = df.loc[ok,'radeg'].values.astype(np.float64)
    dec = df.loc[ok,'dedeg'].values.astype(np.float64)
    sky_ext[name] = make_healpix_map(ra, dec)
    print(f'  {name}: {ok.sum():,} stars')

# Member CSV sky maps
print('\nBuilding sky maps for member catalogs ...')
member_sky = {}; member_n = {}
for key, (path, ra_col, dec_col) in MEMBER_FILES.items():
    npz_key = f'sky_member_{key}'
    if npz_key in D:
        member_sky[key] = D[npz_key].astype(float)
        member_n[key]   = int(np.nansum(member_sky[key]))
        print(f'  {key}: from NPZ ({member_n[key]:,})')
    elif os.path.exists(path):
        mdf = pd.read_csv(path)
        if ra_col in mdf.columns and dec_col in mdf.columns:
            ra  = pd.to_numeric(mdf[ra_col],  errors='coerce').values
            dec = pd.to_numeric(mdf[dec_col], errors='coerce').values
            member_sky[key] = make_healpix_map(ra, dec)
            member_n[key]   = int(np.sum(np.isfinite(ra) & np.isfinite(dec)))
            print(f'  {key}: {member_n[key]:,} stars')
        else:
            print(f'  ✗ {key}: columns not found')
            member_sky[key] = None; member_n[key] = 0
    else:
        print(f'  ✗ {key}: CSV missing')
        member_sky[key] = None; member_n[key] = 0

print('\n✓ All catalog data loaded')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3d — Load Homo_rv checkpoints & CSVs
# ═══════════════════════════════════════════════════════════════
def load_pkl(name):
    p = CKPT_DIR / name
    if not p.exists():
        print(f"  WARNING: {p} not found"); return None
    with open(p, 'rb') as f: d = pickle.load(f)
    print(f"  Loaded {name}  ({p.stat().st_size/1e6:.1f} MB)")
    return d

def load_csv(name):
    p = HOMO_OUT / name
    if not p.exists():
        print(f"  WARNING: {p} not found"); return None
    df = pd.read_csv(p)
    print(f"  Loaded {name}  ({len(df)} rows)")
    return df

print("Loading Homo_rv checkpoints …")
dup   = load_pkl('phase4_v9.pkl')
tch   = load_pkl('phase5_v9.pkl')
gcal  = load_pkl('phase6_v9_allsurveys.pkl')
scal  = load_pkl('phase7_v9.pkl')
nerr  = load_pkl('phase8_v9.pkl')

print("\nLoading Homo_rv CSV outputs …")
csv_fig6   = load_csv('fig6_data.csv')
csv_fig7   = load_csv('fig7_zp_stats.csv')
csv_fig13  = load_csv('fig13_data.csv')
csv_fig13s = load_csv('fig13_stats.csv')
csv_fig17  = load_csv('fig17_drv_srv.csv')
csv_fig17s = load_csv('fig17_stats.csv')
csv_dup    = load_csv('table3_dup.csv')
csv_nf     = load_csv('table4_norm_factors.csv')
csv_drv    = load_csv('table5_drv_stats.csv')
csv_gcal   = load_csv('gaia_cal_coefficients.csv')
csv_zp     = load_csv('gaia_cal_zp_shifts.csv')
csv_scal   = load_csv('survey_cal_coefficients.csv')
print("✓ Homo_rv data loaded")


---
# Part I — Catalog Visualization Figures

## Figure 1 — Sky Coverage Grid (4×4, 16 panels)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 1 — Sky Grid (4×4, white bg, +PRISTINE +CAT panels)
# ═══════════════════════════════════════════════════════════════
if not HEALPY_OK:
    print('Skipping — healpy unavailable')
else:
    L, B, lon_rad, lat_rad, pix_grid = mollweide_grid(NSIDE)
    l_line = L[0, :]

    CMAP_SKY = 'turbo'

    def _plot_hp_white(fig, ax, sky_arr, title, n_label):
        """Plot healpix density map on Mollweide axis with WHITE background."""
        ax.set_facecolor('white')
        if sky_arr is None:
            ax.text(0.5, 0.5, f'{title}\nNo data',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=13, color='gray')
            style_mollweide_white(ax, title, 0)
            return False
        p = sky_arr.copy().astype(float); p[p == 0] = np.nan
        fin = p[np.isfinite(p)]
        if len(fin) == 0:
            ax.text(0.5, 0.5, f'{title}\nAll zero',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=13, color='gray')
            style_mollweide_white(ax, title, 0)
            return False
        vmax = float(np.nanpercentile(fin, 99.5))
        if not np.isfinite(vmax) or vmax < 1:
            vmax = max(float(np.nanmax(fin)), 2.0)
        grid = p[pix_grid].reshape(L.shape)
        im = ax.pcolormesh(lon_rad, lat_rad, grid,
                           cmap=CMAP_SKY,
                           norm=LogNorm(vmin=1, vmax=vmax),
                           shading='auto', rasterized=True)
        cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                          pad=0.04, aspect=40, shrink=0.75, fraction=0.03)
        cb.set_label('Stars / pixel', fontsize=9, fontweight='bold')
        cb.ax.tick_params(labelsize=8)
        # Galactic plane + centre
        ax.plot(-np.radians(l_line), np.zeros_like(l_line),
                '-', color='black', lw=0.8, alpha=0.4, zorder=9)
        ax.plot(0, 0, '+', color='red', ms=10, mew=1.8, zorder=10)
        style_mollweide_white(ax, title, n_label)
        return True

    # ── Build combined sky arrays ─────────────────────────────────────────
    def _hp(key):
        arr = D.get(key)
        return arr.astype(float) if arr is not None else None
    def _ext(name):
        arr = sky_ext.get(name)
        return arr.astype(float) if arr is not None else None
    def _n(arr):
        return int(np.nansum(arr)) if arr is not None else 0

    # Gaia 125M + 33M combined
    sky_125 = D.get('sky_Gaia125M'); sky_33 = D.get('sky_Gaia33M')
    if sky_125 is not None and sky_33 is not None:
        sky_125_33 = sky_125.astype(float) + sky_33.astype(float)
    elif sky_125 is not None: sky_125_33 = sky_125.astype(float)
    elif sky_33 is not None:  sky_125_33 = sky_33.astype(float)
    else: sky_125_33 = None
    n_125_33 = int(np.nansum(sky_125_33)) if sky_125_33 is not None else 0

    # LAMOST combined
    sky_lam_fits = D.get('sky_LAMOST'); sky_lam_ext = sky_ext.get('LAMOST_ext')
    if sky_lam_fits is not None and sky_lam_ext is not None:
        sky_lamost_all = sky_lam_fits.astype(float) + sky_lam_ext.astype(float)
    elif sky_lam_fits is not None: sky_lamost_all = sky_lam_fits.astype(float)
    elif sky_lam_ext is not None:  sky_lamost_all = sky_lam_ext.astype(float)
    else: sky_lamost_all = None
    n_lamost = int(np.nansum(sky_lamost_all)) if sky_lamost_all is not None else 0

    n_sos  = int(np.nansum(D['sky_SOS']))   if 'sky_SOS'  in D else 0
    n_desi = int(np.nansum(D['sky_DESI']))  if 'sky_DESI' in D else 0
    n_sdss = int(np.nansum(D['sky_SDSS']))  if 'sky_SDSS' in D else 0

    # ── 16 panels (4×4) ──────────────────────────────────────────────────
    PANELS_16 = [
        ('hp', 'All (Filtered)',            _hp('sky_full_filt'),
              int(D.get('_n_filtered', np.array([0]))[0])),
        ('hp', 'Gaia — SOS',               _hp('sky_SOS'),         n_sos),
        ('hp', 'Gaia — 125M + 33M',        sky_125_33,             n_125_33),
        ('hp', 'LAMOST (FITS + ext)',       sky_lamost_all,         n_lamost),

        ('hp', 'DESI',                      _hp('sky_DESI'),        n_desi),
        ('hp', 'SDSS / BOSS / SEGUE',      _hp('sky_SDSS'),        n_sdss),
        ('hp', 'APOGEE',                   _ext('APOGEE'),         _n(_ext('APOGEE'))),
        ('hp', 'GALAH',                    _ext('GALAH'),          _n(_ext('GALAH'))),

        ('hp', 'GES',                      _ext('GES'),            _n(_ext('GES'))),
        ('hp', 'RAVE',                     _ext('RAVE'),           _n(_ext('RAVE'))),
        ('hp', 'PRISTINE',                 _hp('sky_PRISTINE'),    _n(_hp('sky_PRISTINE'))),
        ('hp', 'CAT',                      _hp('sky_CAT'),         _n(_hp('sky_CAT'))),

        ('scatter_sgr', 'Sgr Stream',       None, member_n.get('SGR', 0)),
        ('scatter_clusters', 'Stellar Clusters & Dwarfs', None,
         member_n.get('GC',0)+member_n.get('OC',0)+member_n.get('DWG',0)),
        ('empty', '', None, 0),
        ('empty', '', None, 0),
    ]

    NROWS, NCOLS = 4, 4
    fig = plt.figure(figsize=(11*NCOLS, 6.5*NROWS), facecolor='white')
    fig.suptitle('Sky Coverage — All Surveys & Member Catalogs',
                 fontsize=32, fontweight='bold', color='black', y=1.005)

    gs = gridspec.GridSpec(NROWS, NCOLS, figure=fig,
                           hspace=0.25, wspace=0.08,
                           left=0.03, right=0.97, top=0.96, bottom=0.03)

    CLUSTER_STYLES = {
        'GC':  {'c':'#E63946','mk':'o','s':22,'alpha':0.80,'label':'Globular Clusters','zorder':12},
        'OC':  {'c':'#457B9D','mk':'s','s':18,'alpha':0.75,'label':'Open Clusters','zorder':11},
        'DWG': {'c':'#E9C46A','mk':'D','s':28,'alpha':0.85,'label':'Dwarf Galaxies','zorder':13},
    }
    SGR_STYLE = {'c':'#2A9D8F','mk':'^','s':8,'alpha':0.55,'label':'Sgr Stream'}

    for idx, (ptype, title, sky_arr, n_label) in enumerate(PANELS_16):
        row = idx // NCOLS; col = idx % NCOLS

        if ptype == 'empty':
            ax = fig.add_subplot(gs[row, col])
            ax.set_visible(False); continue

        ax = fig.add_subplot(gs[row, col], projection='mollweide')
        ax.set_facecolor('white')

        if ptype == 'hp':
            ok = _plot_hp_white(fig, ax, sky_arr, title, n_label)
            print(f'  [{idx+1:2d}] {title:35s} {"✓" if ok else "✗ no data"}  N={n_label:,}')

        elif ptype == 'scatter_sgr':
            style_mollweide_white(ax, title, n_label)
            ax.plot(-np.radians(l_line), np.zeros_like(l_line),
                    '-', color='black', lw=0.8, alpha=0.4, zorder=9)
            ax.plot(0, 0, '+', color='red', ms=10, mew=1.8, zorder=10)
            sgr_path, sgr_ra_col, sgr_dec_col = MEMBER_FILES.get('SGR', (None,None,None))
            if sgr_path and os.path.exists(sgr_path):
                sgr_df = pd.read_csv(sgr_path)
                ra_s  = pd.to_numeric(sgr_df.get(sgr_ra_col,pd.Series()),errors='coerce').values
                dec_s = pd.to_numeric(sgr_df.get(sgr_dec_col,pd.Series()),errors='coerce').values
                ok_s  = np.isfinite(ra_s) & np.isfinite(dec_s)
                if np.any(ok_s):
                    xm, ym = radec_to_moll(ra_s[ok_s], dec_s[ok_s])
                    ax.scatter(xm, ym, c=SGR_STYLE['c'], marker=SGR_STYLE['mk'],
                               s=SGR_STYLE['s'], alpha=SGR_STYLE['alpha'],
                               linewidths=0, zorder=10, rasterized=True)
                    ax.legend(handles=[Line2D([0],[0], marker=SGR_STYLE['mk'],
                              color='w', markerfacecolor=SGR_STYLE['c'],
                              markersize=8, label=f"Sgr Stream ({ok_s.sum():,})")],
                              loc='lower right', fontsize=9, framealpha=0.85)

        elif ptype == 'scatter_clusters':
            style_mollweide_white(ax, title, n_label)
            ax.plot(-np.radians(l_line), np.zeros_like(l_line),
                    '-', color='black', lw=0.8, alpha=0.4, zorder=9)
            ax.plot(0, 0, '+', color='red', ms=10, mew=1.8, zorder=10)
            legend_h = []
            for cat_key, sty in CLUSTER_STYLES.items():
                cat_path, ra_col, dec_col = MEMBER_FILES.get(cat_key, (None,None,None))
                if cat_path and os.path.exists(cat_path):
                    cdf = pd.read_csv(cat_path)
                    ra_c = pd.to_numeric(cdf.get(ra_col,pd.Series()),errors='coerce').values
                    dec_c = pd.to_numeric(cdf.get(dec_col,pd.Series()),errors='coerce').values
                    ok_c = np.isfinite(ra_c) & np.isfinite(dec_c)
                    if np.any(ok_c):
                        xm, ym = radec_to_moll(ra_c[ok_c], dec_c[ok_c])
                        ax.scatter(xm, ym, c=sty['c'], marker=sty['mk'],
                                   s=sty['s'], alpha=sty['alpha'],
                                   linewidths=0.3, edgecolors='black',
                                   zorder=sty['zorder'], rasterized=True)
                        legend_h.append(Line2D([0],[0], marker=sty['mk'], color='w',
                                        markerfacecolor=sty['c'], markersize=9,
                                        markeredgecolor='black', markeredgewidth=0.4,
                                        label=f"{sty['label']} ({ok_c.sum():,})"))
            if legend_h:
                ax.legend(handles=legend_h, loc='lower right',
                          fontsize=9, framealpha=0.85)

    fig.text(0.5, -0.005,
             'Galactic coordinates (l, b) — Mollweide projection   '
             '|   Red cross = Galactic centre   '
             '|   Black line = Galactic plane',
             ha='center', va='bottom', fontsize=13, fontstyle='italic')

    savefig(fig, 'Fig1_sky_grid')
    plt.show()
    print(f'\n✓ Saved Fig1_sky_grid')


## Figure 2 — Extinction Sky Map

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 2 — Extinction Sky Map
# ═══════════════════════════════════════════════════════════════
if not HEALPY_OK:
    print('Skipping — healpy unavailable')
else:
    ext_ebprp = D.get('ext_ebprp_mean')
    ext_ag    = D.get('ext_ag_mean')

    if ext_ebprp is None or ext_ag is None:
        print('Extinction maps missing — run FITS recompute cell first')
    else:
        ext_ebprp = ext_ebprp.astype(float)
        ext_ag    = ext_ag.astype(float)

        L, B, lon_rad, lat_rad, pix_grid = mollweide_grid(NSIDE)
        l_line = L[0,:]

        fig = plt.figure(figsize=(30, 12), facecolor='white')
        fig.suptitle('Stellar Extinction — Galactic Coordinates\n'
                     r'$E(G_{\rm BP}-G_{\rm RP})$ (left)   ·   $A_G$ (right)',
                     fontsize=26, fontweight='bold', y=1.02)

        gs = gridspec.GridSpec(1, 4, width_ratios=[30,1.2,30,1.2], wspace=0.12)
        axs   = [fig.add_subplot(gs[0,0], projection='mollweide'),
                 fig.add_subplot(gs[0,2], projection='mollweide')]
        axcbs = [fig.add_subplot(gs[0,1]), fig.add_subplot(gs[0,3])]

        panels_ext = [
            (ext_ebprp, r'$E(G_{\rm BP}-G_{\rm RP})$ [mag]', 'afmhot'),
            (ext_ag,    r'$A_G$ [mag]',                       'plasma'),
        ]

        for i, (sky_map, label, cmap_name) in enumerate(panels_ext):
            ax = axs[i]; ax.set_facecolor('white')
            fin = sky_map[np.isfinite(sky_map) & (sky_map>0)]
            if len(fin)==0:
                ax.text(0.5,0.5,f'{label}\nNo data',transform=ax.transAxes,
                        ha='center',va='center',fontsize=16,color='gray')
                ax.set_title(label, fontsize=20, fontweight='bold')
                axcbs[i].set_visible(False); continue

            vmax = float(np.percentile(fin, 99))
            if not np.isfinite(vmax) or vmax<=0: vmax = float(np.nanmax(fin))
            norm_ext = Normalize(vmin=0, vmax=vmax)
            grid = sky_map[pix_grid].reshape(L.shape)
            ax.pcolormesh(lon_rad, lat_rad, grid, cmap=cmap_name,
                          norm=norm_ext, shading='auto', rasterized=True)

            ax.grid(True, alpha=0.3, ls='-', lw=0.5, color='gray')
            ax.plot(-np.radians(l_line), np.zeros_like(l_line),
                    '-', color='white', lw=1.0, alpha=0.5, zorder=9)
            for boff in [5,-5]:
                ax.plot(-np.radians(l_line),
                        np.radians(np.full_like(l_line, float(boff))),
                        '--', color='cyan', lw=0.9, alpha=0.5, zorder=9)
            ax.plot(0, 0, '+', color='lime', ms=14, mew=2.5, zorder=10)
            ax.set_title(label, fontsize=20, fontweight='bold', pad=8)

            lt = np.array([150,120,90,60,30,0,-30,-60,-90,-120,-150])
            ax.set_xticks(np.radians(lt))
            ax.set_xticklabels([f'{int(np.mod(-ll,360))}°' for ll in lt], fontsize=11)
            ax.set_yticks(np.radians([-60,-30,0,30,60]))
            ax.set_yticklabels([f'{b}°' for b in [-60,-30,0,30,60]], fontsize=11)

            med = float(np.nanmedian(fin)); mean = float(np.nanmean(fin))
            ax.text(0.01, 0.02,
                    f'Median={med:.3f}  Mean={mean:.3f}\n'
                    f'Max={float(np.nanmax(fin)):.3f}  N_pix={len(fin):,}',
                    transform=ax.transAxes, fontsize=12, fontweight='bold',
                    va='bottom', ha='left',
                    bbox=dict(boxstyle='round,pad=0.4',facecolor='white',alpha=0.9))

            cb = ColorbarBase(axcbs[i], cmap=plt.get_cmap(cmap_name),
                              norm=norm_ext, orientation='vertical')
            cb.set_label(label, fontsize=14, fontweight='bold', labelpad=8)
            cb.ax.tick_params(labelsize=12)

        savefig(fig, 'Fig2_extinction')
        plt.show()
        print(f'✓ Saved Fig2_extinction')


## Figure 3 — HR Diagram

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 3 — HR Diagram
#   Gray: unfiltered   |   Coloured: filtered   |   Contours: combined external
#   Teff top axis stretched across full x-range with fine ticks
# ═══════════════════════════════════════════════════════════════
hr_u = hr_unfilt.copy(); hr_u[hr_u==0] = np.nan
hr_f = hr_filt.copy();   hr_f[hr_f==0] = np.nan

vmax_u = safe_vmax(hr_u)
vmax_f = safe_vmax(hr_f)
print(f'HR unfilt: bins={int(np.sum(np.isfinite(hr_u))):,}  vmax={vmax_u}')
print(f'HR filt  : bins={int(np.sum(np.isfinite(hr_f))):,}  vmax={vmax_f}')

if vmax_u is None and vmax_f is None:
    print('✗ No HR data — run FITS recompute cell first')
else:
    fig, ax = plt.subplots(figsize=(14, 20), facecolor='white')
    ax.set_facecolor('white')

    # Gray background — unfiltered
    if vmax_u is not None:
        ax.pcolormesh(bprp_edges, mg_edges, hr_u, cmap='Greys',
                      norm=LogNorm(vmin=1, vmax=vmax_u),
                      shading='flat', rasterized=True, alpha=0.35, zorder=2)

    # Turbo foreground — filtered
    im = None
    if vmax_f is not None:
        im = ax.pcolormesh(bprp_edges, mg_edges, hr_f, cmap='turbo',
                           norm=LogNorm(vmin=1, vmax=vmax_f),
                           shading='flat', rasterized=True, alpha=0.80, zorder=3)
        add_density_contours(ax, hr_f, bprp_edges, mg_edges,
                             n_levels=6, color='white', lw=1.0, sigma=4.0)

    # ── External survey contours (all 5 combined) ─────────────────────────
    ext_bprp_all = []; ext_mg_all = []
    for name, df in surveys_ext.items():
        ok = df['teff'].notna() & df['logg'].notna()
        # Need BP-RP and Mg — approximate from Teff if available
        # Use the existing HR bins to build a 2D histogram overlay
        # External surveys in .dat don't have BP-RP/Mg directly,
        # so we plot their Kiel data as contours on the Kiel diagram instead.
        # For HR diagram, overlay contours from the FILTERED catalog itself.
        pass

    # Instead, overlay contours from the main RV + distance catalog
    # using master_cache weighted-average RV stars
    dist_hr = master[COL_DIST]
    wa_hr   = master[COL_WA]
    ok_rv_hr = np.isfinite(dist_hr) & np.isfinite(wa_hr) & (dist_hr > 0)
    n_rv = int(ok_rv_hr.sum())
    if n_rv > 100:
        # Contours from the density of the FILTERED catalog (hr_f)
        # already plotted as white contours above
        # Add black contours from unfiltered for contrast
        if vmax_u is not None:
            add_density_contours(ax, hr_u, bprp_edges, mg_edges,
                                 n_levels=4, color='black', lw=0.6, sigma=5.0)

    if im is not None:
        cbar = plt.colorbar(im, ax=ax, pad=0.02, aspect=42)
        cbar.set_label('Stars — Filtered (turbo)', fontsize=20, fontweight='bold')
        cbar.ax.tick_params(labelsize=15)

    ax.invert_yaxis()
    ax.set_xlabel(r'$(G_{\rm BP}-G_{\rm RP})_0$ [mag]',
                  fontsize=26, fontweight='bold', labelpad=12)
    ax.set_ylabel(r'$M_G$ [mag]', fontsize=26, fontweight='bold', labelpad=12)
    ax.set_title(
        f'HR Diagram\nGray: All N={n_total:,}   |   Colour: RUWE-filtered N={n_filt:,}',
        fontsize=20, fontweight='bold', pad=12)
    ax.set_xlim(-1.5, 5.5); ax.set_ylim(16.0, -8.0)
    ax.tick_params(which='major', length=10, width=2.0, direction='in',
                   top=True, right=True, labelsize=18)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='minor', length=5, width=1.2, direction='in',
                   top=True, right=True)
    ax.grid(True, alpha=0.15, ls='--', lw=0.5)

    # ── Teff top axis — stretched across full x-range ─────────────────────
    teff_m = D.get('teff_bprp_mean')
    if teff_m is not None and np.any(np.isfinite(teff_m)):
        bprp_c = (bprp_edges[:-1]+bprp_edges[1:])/2.0
        valid  = np.isfinite(teff_m)
        bprp_v = bprp_c[valid]; teff_v = teff_m[valid]

        # Finer Teff ticks every ~500 K, spanning full range
        teff_targets = [10000, 9000, 8500, 8000, 7500, 7000, 6500, 6000,
                        5500, 5000, 4800, 4500, 4200, 4000, 3700, 3400, 3200]
        t_pos = []; t_lbl = []; last_pos = -999.0
        for t in teff_targets:
            if len(teff_v) == 0: break
            idx = np.argmin(np.abs(teff_v - t))
            pos = float(bprp_v[idx]); diff = abs(teff_v[idx] - t)
            if -1.3 <= pos <= 5.3 and diff < 800 and (pos - last_pos) >= 0.30:
                t_pos.append(pos)
                t_lbl.append(f'{t:,}')
                last_pos = pos

        if t_pos:
            ax_top = ax.twiny()
            ax_top.set_xlim(ax.get_xlim())
            ax_top.set_xticks(t_pos)
            ax_top.set_xticklabels(t_lbl, fontsize=12, fontweight='bold',
                                    rotation=45, ha='left')
            ax_top.tick_params(which='major', length=10, width=1.8,
                               direction='out', pad=6, labelsize=12)
            ax_top.xaxis.set_minor_locator(NullLocator())
            ax_top.set_xlabel(r'$T_{\rm eff}$ [K]',
                              fontsize=20, fontweight='bold', labelpad=18)

    plt.tight_layout(); plt.subplots_adjust(top=0.82)
    savefig(fig, 'Fig3_HR_diagram')
    plt.show()
    print(f'✓ Saved Fig3_HR_diagram')


## Figure 4 — Kiel Diagram (single panel)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 4 — Kiel Diagram (single panel: gray unfilt + coloured filt)
# ═══════════════════════════════════════════════════════════════
ku = kiel_unfilt.copy(); ku[ku==0] = np.nan
kf = kiel_filt.copy();   kf[kf==0] = np.nan

fig, ax = plt.subplots(figsize=(14, 13), facecolor='white')
ax.set_facecolor('white')

# Gray background — unfiltered
vmax_ku = safe_vmax(ku)
if vmax_ku:
    ax.pcolormesh(teff_edges, logg_edges, ku, cmap='Greys',
                  norm=LogNorm(vmin=1, vmax=vmax_ku),
                  shading='flat', rasterized=True, alpha=0.35, zorder=2)

# Turbo foreground — filtered
im_kf = None
vmax_kf = safe_vmax(kf)
if vmax_kf:
    im_kf = ax.pcolormesh(teff_edges, logg_edges, kf, cmap='turbo',
                           norm=LogNorm(vmin=1, vmax=vmax_kf),
                           shading='flat', rasterized=True, alpha=0.80, zorder=3)
    add_density_contours(ax, kf, teff_edges, logg_edges,
                         n_levels=6, color='white', lw=0.9, sigma=3.5)
    cb = plt.colorbar(im_kf, ax=ax, pad=0.02, aspect=42)
    cb.set_label('Stars — Filtered', fontsize=18, fontweight='bold')
    cb.ax.tick_params(labelsize=14)

# Black contours from unfiltered for contrast
if vmax_ku:
    add_density_contours(ax, ku, teff_edges, logg_edges,
                         n_levels=4, color='black', lw=0.6, sigma=5.0)

ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel(r'$T_{\rm eff}$ [K]', fontsize=24, fontweight='bold')
ax.set_ylabel(r'$\log g$ [dex]', fontsize=24, fontweight='bold')
ax.set_title(f'Kiel Diagram\nGray: N={n_total:,}  |  Colour: RUWE-filtered N={n_filt:,}',
             fontsize=20, fontweight='bold', pad=10)
ax.set_xlim(10000, 3000); ax.set_ylim(5.5, -0.5)
ax.tick_params(which='major', length=10, width=2.0, direction='in',
               top=True, right=True, labelsize=18)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='minor', length=5, width=1.2, direction='in',
               top=True, right=True)
ax.grid(True, alpha=0.15, ls='--', lw=0.5)

plt.tight_layout()
savefig(fig, 'Fig4_Kiel')
plt.show()
print(f'✓ Saved Fig4_Kiel')


## Figure 5 — RV vs Distance Phase-Space

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 5 — RV vs Distance Phase-Space
#   Main catalog density + own contours only (no cluster members)
# ═══════════════════════════════════════════════════════════════
DIST_MAX = 150.0; RV_MAX = 500.0
NBINS_D  = 300;   NBINS_R = 200
DR = (0, DIST_MAX); RR = (-RV_MAX, RV_MAX)
cmap_rv = make_magma_cmap()

dist_m = master[COL_DIST]
wa_m   = master[COL_WA]
zp_m   = master[COL_ZP]

ok_wa = np.isfinite(dist_m) & np.isfinite(wa_m) & (dist_m>0)
ok_zp = np.isfinite(dist_m) & np.isfinite(zp_m) & (dist_m>0)
d_wa = dist_m[ok_wa]; r_wa = wa_m[ok_wa]
d_zp = dist_m[ok_zp]; z_zp = zp_m[ok_zp]
print(f'WA pairs: {len(d_wa):,}   ZP pairs: {len(d_zp):,}')

def binned_median(x, y, n_bins=150, xmax=DIST_MAX, min_n=5):
    edges = np.linspace(0, xmax, n_bins)
    cx = 0.5*(edges[:-1]+edges[1:])
    med = np.full(len(cx), np.nan)
    idx = np.digitize(x, edges)-1
    for i in range(len(cx)):
        m = idx==i
        if m.sum()>=min_n: med[i]=np.median(y[m])
    return cx, med

fig, ax = plt.subplots(figsize=(14,10), facecolor='white')
ax.set_facecolor('white')

# 2D density histogram
H, xe, ye = np.histogram2d(d_wa, r_wa, bins=[NBINS_D,NBINS_R],
                            range=[list(DR),list(RR)])
H = H.T; Hp = H.copy(); Hp[Hp==0] = np.nan
im = ax.pcolormesh(xe, ye, Hp, cmap=cmap_rv,
                   norm=mcolors.LogNorm(vmin=1, vmax=np.nanmax(Hp)),
                   rasterized=True)
cb = plt.colorbar(im, ax=ax, pad=0.02, aspect=30)
cb.set_label('Star Count (log)', fontsize=14, fontweight='bold')

# Contours from the main catalog density itself
Hs = gaussian_filter(H.astype(float), sigma=3.0)
occ = Hs[Hs>0]
if len(occ)>10:
    levs_w = sorted(set(np.percentile(occ, [40, 60, 78, 90, 97])))
    levs_w = [l for l in levs_w if l > 0]
    if len(levs_w) >= 2:
        xca = 0.5*(xe[:-1]+xe[1:]); yca = 0.5*(ye[:-1]+ye[1:])
        Xma, Yma = np.meshgrid(xca, yca)
        ax.contour(Xma, Yma, Hs, levels=levs_w, colors='white',
                   linewidths=0.8, alpha=0.65, linestyles='--', zorder=20)

# Median overlays
if len(d_zp)>200:
    cx, my = binned_median(d_zp, z_zp); ok = np.isfinite(my)
    ax.plot(cx[ok], my[ok], color='#FFD700', lw=2.5, ls='--',
            alpha=0.95, label='Median ZP', zorder=25)
if len(d_wa)>200:
    cx, my = binned_median(d_wa, r_wa); ok = np.isfinite(my)
    ax.plot(cx[ok], my[ok], color='#00FFFF', lw=2.0, ls='--',
            alpha=0.85, label='Median Weighted Avg', zorder=24)

handles = [
    Line2D([0],[0], color='#FFD700', ls='--', lw=2.5, label='Median ZP'),
    Line2D([0],[0], color='#00FFFF', ls='--', lw=2.0, label='Median Weighted Avg'),
    Line2D([0],[0], color='white',   ls='--', lw=1.0, label='Catalog Density Contours'),
]
ax.legend(handles=handles, loc='upper right', fontsize=11,
          framealpha=0.92, edgecolor='gray')

ax.set_xlabel('Distance (kpc)', fontsize=18, fontweight='bold')
ax.set_ylabel(r'Radial Velocity (km s$^{-1}$)', fontsize=18, fontweight='bold')
ax.set_title('RV vs Distance Phase-Space', fontsize=20, fontweight='bold')
ax.set_xlim(DR); ax.set_ylim(RR)
ax.tick_params(which='major', length=8, width=1.8, direction='in',
               top=True, right=True, labelsize=14)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='minor', length=5, width=1.0, direction='in',
               top=True, right=True)

plt.tight_layout()
savefig(fig, 'Fig5_RV_Distance')
plt.show()
print(f'✓ Saved Fig5_RV_Distance')


## Figure 6 — Error Analysis (2-panel)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Figure 6 — Error Analysis (2 panels: N vs Dist + Median Error vs Dist)
#   (Normalized density panel removed per request)
# ═══════════════════════════════════════════════════════════════
dist_e = master[COL_DIST]
wa_err = master[COL_WA_ERR]
rv_err = master[COL_RV_ERR]
zp_err = master[COL_ZP_ERR]
rv_raw = master[COL_RV]
nmeas  = master[COL_NMEAS]
rvsn   = master[COL_RVSN]
ruwe_e = master[COL_RUWE]
ok_dist = np.isfinite(dist_e) & (dist_e>0)

def bstat_err(x, y, n_bins=120, xmax=DIST_MAX, min_n=5):
    edges = np.linspace(0, xmax, n_bins)
    cx = 0.5*(edges[:-1]+edges[1:])
    med = np.full(len(cx), np.nan)
    idx = np.digitize(x, edges)-1
    for i in range(len(cx)):
        vals = y[idx==i]
        if np.sum(np.isfinite(vals))>=min_n: med[i]=np.nanmedian(vals)
    return cx, med

def bcount(x, mask, n_bins=150, xmax=DIST_MAX):
    edges = np.linspace(0, xmax, n_bins)
    cx = 0.5*(edges[:-1]+edges[1:])
    x_ok = x[mask]; idx = np.digitize(x_ok, edges)-1
    cnt = np.array([np.sum(idx==i) for i in range(len(cx))], dtype=float)
    cnt[cnt==0] = np.nan; return cx, cnt

def bsum_f(x, y, mask, n_bins=150, xmax=DIST_MAX):
    edges = np.linspace(0, xmax, n_bins)
    cx = 0.5*(edges[:-1]+edges[1:])
    x_ok = x[mask]; y_ok = y[mask]; idx = np.digitize(x_ok, edges)-1
    s = np.array([np.nansum(y_ok[idx==i]) for i in range(len(cx))], dtype=float)
    s[s==0] = np.nan; return cx, s

fig, axes = plt.subplots(2, 1, figsize=(14, 12),
                         gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.30})

# ── Panel 1: N (log) vs Distance ─────────────────────────────────────────
ax = axes[0]
ok_rv = ok_dist & np.isfinite(rv_raw)
cx, cnt_rv = bcount(dist_e, ok_rv)
ax.plot(cx, cnt_rv, color='#DC143C', lw=2.0, alpha=0.9, label='Raw RV count')

ok_nm = ok_dist & np.isfinite(nmeas) & (nmeas>0)
cx, sum_nm = bsum_f(dist_e, nmeas, ok_nm)
ax.plot(cx, sum_nm, color='#6495ED', lw=2.0, alpha=0.9,
        label=r'$\Sigma\,n_{\rm meas}$')

bad = ok_dist & ((np.isfinite(ruwe_e)&(ruwe_e>1.4))|
                  (np.isfinite(rvsn)&(rvsn<3.0)))
cx, cnt_out = bcount(dist_e, bad)
ax.plot(cx, cnt_out, color='#77DD77', lw=1.8, alpha=0.85, label='Outlier count')

ax.set_yscale('log')
ax.set_xlabel('Distance (kpc)', fontsize=16, fontweight='bold')
ax.set_ylabel(r'$N$ (log)', fontsize=16, fontweight='bold')
ax.set_xlim(0, DIST_MAX)
ax.legend(fontsize=11, loc='upper right', framealpha=0.92, edgecolor='gray')
ax.grid(True, alpha=0.2, which='both'); ax.minorticks_on()
add_panel_label(ax, '(a)')

# ── Panel 2: Mean error vs Distance ──────────────────────────────────────
ax = axes[1]
for col, color, ls, label in [
    (COL_WA_ERR, '#4169E1', '-',  'Median Weighted Avg err'),
    (COL_ZP_ERR, '#FFD700', '--', 'Median ZP err'),
    (COL_RV_ERR, '#DC143C', '-.', 'Median RV err'),
]:
    e = master[col]
    ok = ok_dist & np.isfinite(e) & (e>0)
    if ok.sum()<200: continue
    cx, me = bstat_err(dist_e[ok], e[ok])
    ok2 = np.isfinite(me)
    ax.plot(cx[ok2], me[ok2], color=color, lw=2.0, ls=ls, alpha=0.9, label=label)

ax.set_xlabel('Distance (kpc)', fontsize=16, fontweight='bold')
ax.set_ylabel(r'Median Error (km s$^{-1}$)', fontsize=16, fontweight='bold')
ax.set_xlim(0, DIST_MAX)
ax.legend(fontsize=11, loc='upper right', framealpha=0.92, edgecolor='gray')
ax.grid(True, alpha=0.2, axis='y'); ax.minorticks_on()
add_panel_label(ax, '(b)')

savefig(fig, 'Fig6_error_analysis')
plt.show()
print(f'✓ Saved Fig6_error_analysis')


---
# Part II — RV Normalisation Figures

## Fig 7 — DUP: Normalised ΔRV/σ distributions

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
X_LIM      = (-12, 12)    # x-axis range
Y_LIM      = (0, 0.55)    # y-axis range  (None = auto)
N_GAUSS_PTS = 600         # smoothness of N(0,1) reference
ALPHA       = 0.85
FIG_SIZE    = (10, 5.5)
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIG_SIZE)

xg = np.linspace(*X_LIM, N_GAUSS_PTS)
ax.plot(xg, norm.pdf(xg, 0, 1), 'k--', lw=2, alpha=0.6, label='N(0,1)', zorder=10)

if csv_fig6 is not None:
    bc = csv_fig6['bin_center'].values
    surv_cols = [c for c in csv_fig6.columns if c != 'bin_center']
    for s in sorted(surv_cols):
        h = csv_fig6[s].values
        ok = np.isfinite(h)
        if not np.any(ok): continue
        # stats from table if available
        if csv_dup is not None:
            row = csv_dup[csv_dup['Survey']==s]
            nf  = row['NormMAD'].values[0] if len(row) else np.nan
            np_ = row['N_pairs'].values[0] if len(row) else 0
            lbl = f'{s}  (normMAD={nf:.2f}, N={int(np_):,})' if np.isfinite(nf) else s
        else:
            lbl = s
        ax.step(bc[ok], h[ok], where='mid', lw=2.0, color=_c(s), alpha=ALPHA, label=lbl)
elif dup is not None:
    # Fall back to pk data
    bins = np.linspace(*X_LIM, 161)
    bc2 = 0.5*(bins[:-1]+bins[1:])
    for s in sorted(dup.keys()):
        nd = dup[s]['norm_diffs']
        nd = nd[(nd >= X_LIM[0]) & (nd <= X_LIM[1])]
        if len(nd)<50: continue
        h, _ = np.histogram(nd, bins=bins, density=True)
        nf   = dup[s].get('norm_mad', np.nan)
        lbl  = f'{s}  (normMAD={nf:.2f}, N={len(nd):,})'
        ax.step(bc2, h, where='mid', lw=2.0, color=_c(s), alpha=ALPHA, label=lbl)

ax.set_xlim(*X_LIM)
if Y_LIM: ax.set_ylim(*Y_LIM)
ax.set_xlabel(r'$\Delta$RV / $\sqrt{\sigma_1^2+\sigma_2^2}$', fontsize=13)
ax.set_ylabel('Normalised density', fontsize=13)
ax.set_title('DUP Method: Normalised RV Differences', fontsize=13)
ax.legend(loc='upper right', ncol=1)
ax.grid(True)
add_panel_label(ax, '(a)')
fig.tight_layout()
savefig(fig, 'homo_fig6_dup_normdiff')
plt.show()


## Fig 8 — ΔRV histograms before / after ZP correction

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
X_LIM   = (-18, 18)   # km/s range
N_BINS  = 160
ALPHA   = 0.85
FIG_SIZE = (16, 6.5)
# ─────────────────────────────────────────────────────────────────────────────

if gcal is not None and 'per_survey' in gcal:
    # Reconstruct ΔRV arrays from per_survey data in phase6 pkl
    # per_survey[key]['ys'] = (ΔRV - ZP), per_survey[key]['zp'] = ZP_shift
    ps   = gcal['per_survey']
    bins = np.linspace(*X_LIM, N_BINS+1)

    # Collect per-survey ΔRV (before = ys + zp, after = ys)
    per_surv_before = {}; per_surv_after = {}
    for key, sv in ps.items():
        sn = sv['surv']
        zp = sv.get('zp', 0.0)
        before = sv['ys'] + zp       # raw ΔRV (ZP already subtracted when stored)
        after  = sv['ys']            # centred
        if sn not in per_surv_before:
            per_surv_before[sn] = []; per_surv_after[sn] = []
        per_surv_before[sn].append(before)
        per_surv_after[sn].append(after)
    per_surv_before = {s: np.concatenate(v) for s,v in per_surv_before.items()}
    per_surv_after  = {s: np.concatenate(v) for s,v in per_surv_after.items()}

    fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE, sharey=False)
    labels = ['Before ZP correction', 'After ZP correction']
    for ax, data_dict, lbl in zip(axes,
                                   [per_surv_before, per_surv_after], labels):
        ax.axvline(0, color='k', ls=':', lw=1, alpha=0.4)
        for s in sorted(data_dict.keys()):
            d = data_dict[s]
            d = d[(d >= X_LIM[0]) & (d <= X_LIM[1])]
            if len(d)<30: continue
            h, _ = np.histogram(d, bins=bins, density=True)
            bc   = 0.5*(bins[:-1]+bins[1:])
            med  = float(np.median(d))
            mad  = float(np.median(np.abs(d-med)))
            ax.step(bc, h, where='mid', lw=2.0, color=_c(s), alpha=ALPHA,
                    label=f'{s}  med={med:+.2f}  MAD={mad:.2f}')
        ax.set_xlabel(r'$\Delta$RV = RV$_{\rm Gaia}$ − RV$_{\rm survey}$  (km/s)', fontsize=12)
        ax.set_ylabel('Normalised density', fontsize=12)
        ax.set_title(lbl, fontsize=12)
        ax.set_xlim(*X_LIM)
        ax.legend(fontsize=7.5, loc='upper right'); ax.grid(True)

    add_panel_label(axes[0], '(a)'); add_panel_label(axes[1], '(b)')
    fig.suptitle(r'$\Delta$RV Histograms  (Fig. 7)', fontsize=13, y=1.01)
    fig.tight_layout()
    savefig(fig, 'homo_fig7_drv_zp')
    plt.show()
else:
    print("Phase-6 per_survey data not found — cannot rebuild Fig 7 from pkl.")
    if csv_fig7 is not None:
        display(csv_fig7)


## Fig 9 — ΔRV vs 6 stellar parameters (Gaia calibration)

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
DRV_YLIM  = (-5, 5)     # km/s
N_BINS    = 35
MIN_COUNT = 15
ALPHA     = 0.9
FIG_SIZE  = (20, 11)
CI_ALPHA  = 0.18         # Bevington confidence-band fill opacity
# ─────────────────────────────────────────────────────────────────────────────

SIX_PARAMS = [
    ('Gmag', 'G magnitude (mag)'),
    ('FeH' , '[Fe/H] (dex)'),
    ('Teff', r'$T_{\rm eff}$ (K)'),
    ('logg', r'$\log g$ (dex)'),
    ('SNR' , 'S/N'),
    ('RV'  , 'RV  (km/s)'),
]
PARAM_TO_EQ = {'Gmag':'Eq5', 'FeH':'Eq6', 'Teff':'Eq7'}

if gcal is None:
    print("Phase-6 checkpoint not available — skipping Fig 8.")
else:
    ps       = gcal.get('per_survey', {})
    coeffs_d = gcal.get('coefficients', {})
    cov_d    = gcal.get('covariances',  {})

    fig, axes = plt.subplots(2, 3, figsize=FIG_SIZE)
    axes = axes.flatten()
    panel_labels = ['(a)','(b)','(c)','(d)','(e)','(f)']

    for pi, (pname, xlabel) in enumerate(SIX_PARAMS):
        ax = axes[pi]
        eq = PARAM_TO_EQ.get(pname)

        # Collect per-survey binned data from per_survey
        for sname in sorted(set(sv['surv'] for sv in ps.values())):
            # Gather all xs/ys for this survey across equations that share pname
            xs_all = []; ys_all = []
            for key, sv in ps.items():
                if sv['surv'] == sname and sv['param'] == pname:
                    zp = sv.get('zp', 0.0)
                    xs_all.append(sv['xs'])
                    ys_all.append(sv['ys'] + zp)  # restore raw ΔRV
            if not xs_all: continue
            xs = np.concatenate(xs_all); ys = np.concatenate(ys_all)
            bc, meds, mads = bin_stat(xs, ys, n_bins=N_BINS, min_count=MIN_COUNT)
            if len(bc)==0: continue
            ax.plot(bc, meds, '-o', ms=3, lw=1.8, color=_c(sname), alpha=ALPHA, label=sname)
            ax.fill_between(bc, meds-mads, meds+mads, color=_c(sname), alpha=0.1)

        # Global polynomial fit + Bevington CI
        if eq and eq in coeffs_d:
            all_xs = np.concatenate([sv['xs'] for k,sv in ps.items() if sv['param']==pname])
            x_fit  = np.linspace(np.percentile(all_xs,1), np.percentile(all_xs,99), 400)
            y_fit  = np.polyval(coeffs_d[eq], x_fit)
            ax.plot(x_fit, y_fit, 'k--', lw=2.2, alpha=0.8, label='Global fit', zorder=10)
            cov = cov_d.get(eq)
            if cov is not None:
                delta = np.array([_poly_err(coeffs_d[eq], cov, xv) for xv in x_fit])
                ok = np.isfinite(delta)
                ax.fill_between(x_fit[ok], (y_fit-delta)[ok], (y_fit+delta)[ok],
                                color='k', alpha=CI_ALPHA, label='Bevington 68% CI')

        ax.axhline(0, color='gray', ls=':', lw=0.9, alpha=0.5)
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(r'$\Delta$RV  (km/s)', fontsize=11)
        ax.set_title(f'$\\Delta$RV vs {pname}', fontsize=12)
        ax.set_ylim(*DRV_YLIM)
        ax.legend(fontsize=7, loc='upper right', ncol=2); ax.grid(True)
        add_panel_label(ax, panel_labels[pi])

    fig.suptitle(r'$\Delta$RV vs Stellar Parameters  (Fig. 8)  —  shaded = Bevington 68% CI',
                 fontsize=12, y=1.005)
    fig.tight_layout()
    savefig(fig, 'homo_fig8_drv_params')
    plt.show()


## Figs 10–12 — Gaia internal calibration: before/after per equation

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
DRV_YLIM  = (-3.0, 3.0)   # km/s for main panels
N_BINS    = 25
MIN_COUNT = 10
CI_ALPHA  = 0.20
ROW_RATIO = [3, 1]         # main panel : N-count panel height ratio
# ─────────────────────────────────────────────────────────────────────────────

if gcal is None:
    print("Phase-6 checkpoint not found.")
else:
    ps       = gcal.get('per_survey', {})
    coeffs_d = gcal.get('coefficients', {})
    cov_d    = gcal.get('covariances',  {})

    for eq_name, coeffs in coeffs_d.items():
        keys = [k for k in ps if k.startswith(eq_name+'_')]
        if not keys: continue
        param = ps[keys[0]]['param']
        cov   = cov_d.get(eq_name)
        surv_list = [ps[k]['surv'] for k in keys]
        n_s = len(surv_list)

        fig = plt.figure(figsize=(max(4.2*n_s, 8), 6))
        gs  = gridspec.GridSpec(2, n_s, height_ratios=ROW_RATIO,
                                hspace=0.07, wspace=0.28)

        for si, k in enumerate(keys):
            sv   = ps[k]
            sn   = sv['surv']
            xs_  = sv['xs']
            ys_  = sv['ys']           # already ZP-subtracted (centred)
            zp   = sv.get('zp', 0.0)
            ys_before = ys_ + zp

            ax_m = fig.add_subplot(gs[0, si])
            ax_n = fig.add_subplot(gs[1, si], sharex=ax_m)

            # Before
            bc,meds,mads,ns = bin_stat_counts(xs_, ys_before, n_bins=N_BINS, min_count=MIN_COUNT)
            if len(bc)==0: continue
            ax_m.plot(bc, meds, 'o-', ms=4, lw=1.8, color='#2166ac', alpha=0.9, label='Before')
            ax_m.fill_between(bc, meds-mads, meds+mads, color='#2166ac', alpha=0.15)

            # After
            bc2,meds2,mads2,ns2 = bin_stat_counts(xs_, ys_, n_bins=N_BINS, min_count=MIN_COUNT)
            if len(bc2)>0:
                ax_m.plot(bc2, meds2, 's-', ms=4, lw=1.8, color='#b2182b', alpha=0.9, label='After')
                ax_m.fill_between(bc2, meds2-mads2, meds2+mads2, color='#b2182b', alpha=0.15)

            # Polynomial + CI
            x_fit = np.linspace(bc.min(), bc.max(), 300)
            y_fit = np.polyval(coeffs, x_fit)
            ax_m.plot(x_fit, y_fit, '--', color='#444', lw=2.0, label='Fit')
            if cov is not None:
                delta = np.array([_poly_err(coeffs, cov, xv) for xv in x_fit])
                ok = np.isfinite(delta)
                ax_m.fill_between(x_fit[ok], (y_fit-delta)[ok], (y_fit+delta)[ok],
                                  color='gray', alpha=CI_ALPHA, label='Fit CI')
                # Error bars at bin centres
                d_bc = np.array([_poly_err(coeffs, cov, xv) for xv in bc])
                ok_b = np.isfinite(d_bc)
                if np.any(ok_b):
                    yf_bc = np.polyval(coeffs, bc)
                    ax_m.errorbar(bc[ok_b], yf_bc[ok_b], yerr=d_bc[ok_b],
                                  fmt='none', ecolor='#444', elinewidth=1.4,
                                  capsize=3, capthick=1.0, alpha=0.7)

            ax_m.axhline(0, color='black', ls='-', lw=0.7, alpha=0.5)
            ax_m.set_ylim(*DRV_YLIM)
            ax_m.set_ylabel(r'$\Delta$RV  (km/s)', fontsize=10)
            ax_m.set_title(sn, fontsize=11, fontweight='bold')
            ax_m.legend(fontsize=7, loc='upper right')
            ax_m.grid(True)
            plt.setp(ax_m.get_xticklabels(), visible=False)

            # N in bins
            if len(ns)>0:
                ax_n.plot(bc, ns, 'k.-', ms=3, lw=1.2)
                ax_n.set_yscale('log')
                ax_n.set_ylabel('N', fontsize=9)
            ax_n.set_xlabel(param, fontsize=10)
            ax_n.grid(True)

        fig.suptitle(f'Gaia Calibration {eq_name}: $\Delta$RV vs {param}  (Figs. 9–11)\n'
                     f'Blue=Before | Red=After | Dashed=Polynomial fit ± Bevington CI',
                     fontsize=11, y=1.02)
        fig.tight_layout()
        savefig(fig, f'homo_gaia_cal_{eq_name}_{param}')
        plt.show()


## Fig 13 — Survey calibration (Eq. 8): ΔRV before/after per survey

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
DRV_YLIM  = (-3.0, 3.0)
N_BINS    = 25
MIN_COUNT = 10
ALPHA     = 0.88
FIG_SIZE_PER_PARAM = (5.5, 4.5)   # per parameter panel
# ─────────────────────────────────────────────────────────────────────────────

param_labels = {
    'Teff': r'$T_{\rm eff}$ (K)',
    'logg': r'$\log g$ (dex)',
    'FeH' : '[Fe/H] (dex)',
    'SNR' : 'S/N',
    'RV'  : 'RV  (km/s)',
    'Gmag': 'G magnitude (mag)',
}
param_list = ['Teff','logg','FeH','SNR','RV','Gmag']

if scal is None:
    print("Phase-7 checkpoint not found — skipping Fig 12.")
else:
    for surv, sres in sorted(scal.items()):
        if not isinstance(sres, dict): continue
        if 'diag' not in sres or 'fits' not in sres: continue
        df_diag   = sres['diag']
        fits_dict = sres.get('fits', {})
        if len(df_diag)<50 or not fits_dict: continue

        # Compute corrected ΔRV for each star
        drv_before = df_diag['drv'].values.astype(float).copy()
        drv_after  = drv_before.copy()
        for split_name, fdata in fits_dict.items():
            fn = fdata['feat_names']; cf = fdata['coeffs']
            sl = split_name.lower()
            if   'dwarf' in sl: msk = df_diag.get('logg', pd.Series([4.0]*len(df_diag))).values > 3.5
            elif 'giant' in sl: msk = df_diag.get('logg', pd.Series([4.0]*len(df_diag))).values <= 3.5
            elif 'cool'  in sl: msk = df_diag.get('Teff', pd.Series([5000]*len(df_diag))).values < 6200
            elif 'hot'   in sl: msk = df_diag.get('Teff', pd.Series([5000]*len(df_diag))).values >= 6200
            else:               msk = np.ones(len(df_diag), bool)
            msk = np.array(msk, dtype=bool)
            feat_map = {'Teff':df_diag['Teff'].values if 'Teff' in df_diag.columns else np.zeros(len(df_diag)),
                        'Teff2':(df_diag['Teff'].values**2) if 'Teff' in df_diag.columns else np.zeros(len(df_diag)),
                        'logg':df_diag['logg'].values if 'logg' in df_diag.columns else np.zeros(len(df_diag)),
                        'FeH':df_diag['FeH'].values if 'FeH' in df_diag.columns else np.zeros(len(df_diag)),
                        'SNR':df_diag['SNR'].values if 'SNR' in df_diag.columns else np.zeros(len(df_diag)),
                        'RV':df_diag['RV'].values if 'RV' in df_diag.columns else np.zeros(len(df_diag)),
                        'intercept':np.ones(len(df_diag))}
            for idx in np.where(msk)[0]:
                corr = sum(float(c)*float(feat_map.get(f,np.zeros(len(df_diag)))[idx])
                           for c,f in zip(cf,fn) if np.isfinite(c))
                drv_after[idx] = drv_before[idx] - corr

        avail = [p for p in param_list
                 if p in df_diag.columns and df_diag[p].notna().sum()>50]
        if not avail: continue
        n_cols = min(len(avail), 3)
        n_rows_g = (len(avail)+n_cols-1)//n_cols
        W = FIG_SIZE_PER_PARAM[0]*n_cols; H = FIG_SIZE_PER_PARAM[1]*n_rows_g

        fig = plt.figure(figsize=(W, H))
        gs_out = gridspec.GridSpec(n_rows_g, n_cols, hspace=0.45, wspace=0.30)
        for pi, pname in enumerate(avail):
            gs_in = gridspec.GridSpecFromSubplotSpec(2,1,subplot_spec=gs_out[pi//n_cols,pi%n_cols],
                                                     height_ratios=[3,1], hspace=0.08)
            ax_m = fig.add_subplot(gs_in[0])
            ax_n = fig.add_subplot(gs_in[1], sharex=ax_m)
            x    = df_diag[pname].values.astype(float)
            bc,meds,mads,ns = bin_stat_counts(x, drv_before, n_bins=N_BINS, min_count=MIN_COUNT)
            if len(bc)==0: continue
            ax_m.plot(bc, meds, 'o-', ms=3, lw=1.6, color='#2166ac', alpha=ALPHA, label='Before')
            ax_m.fill_between(bc, meds-mads, meds+mads, color='#2166ac', alpha=0.12)
            bc2,meds2,mads2,_ = bin_stat_counts(x, drv_after, n_bins=N_BINS, min_count=MIN_COUNT)
            if len(bc2)>0:
                ax_m.plot(bc2, meds2, 's-', ms=3, lw=1.6, color='#b2182b', alpha=ALPHA, label='After')
                ax_m.fill_between(bc2, meds2-mads2, meds2+mads2, color='#b2182b', alpha=0.12)
            ax_m.axhline(0, color='k', ls='-', lw=0.7, alpha=0.5)
            ax_m.set_ylim(*DRV_YLIM); ax_m.set_title(pname, fontsize=10)
            ax_m.set_ylabel(r'$\Delta$RV  (km/s)', fontsize=9)
            ax_m.legend(fontsize=7, loc='upper right'); ax_m.grid(True)
            plt.setp(ax_m.get_xticklabels(), visible=False)
            if len(ns)>0:
                ax_n.plot(bc, ns, 'k.-', ms=2, lw=0.9)
                ax_n.set_yscale('log'); ax_n.set_ylabel('N', fontsize=8)
            ax_n.set_xlabel(param_labels.get(pname,pname), fontsize=9); ax_n.grid(True)

        fig.suptitle(f'Survey Calibration (Eq. 8): {surv}  —  N={len(df_diag):,}\n'
                     f'Blue=Before | Red=After', fontsize=12, y=1.01)
        fig.tight_layout()
        savefig(fig, f'homo_fig12_{surv}')
        plt.show()


## Fig 14 — Normalised RV error distributions

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
X_LIM    = (0.0, 4.0)    # km/s
X_LABEL  = r'Normalised RV error  $\sigma \times f$  (km/s)'
ALPHA    = 0.88
FIG_SIZE = (18, 6.5)
BINS     = np.linspace(0, 4, 101)
# ─────────────────────────────────────────────────────────────────────────────

bc = 0.5*(BINS[:-1]+BINS[1:])

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
ax_h, ax_c = axes

source_label = 'Pipeline × DUP/TCH factor'

def _add_fig13_survey(s, ea, f_val, ax_hist, ax_cum, alpha=ALPHA):
    ep  = ea[(ea >= X_LIM[0]) & (ea <= X_LIM[1]) & np.isfinite(ea)]
    if len(ep) < 20: return
    med = float(np.median(ep))
    f_s = f'{f_val:.2f}' if np.isfinite(f_val) else 'n/a'
    h, _ = np.histogram(ep, bins=BINS, density=True)
    cum  = np.cumsum(h) * (BINS[1]-BINS[0])
    ax_hist.step(bc, h,   where='mid', lw=2.0, color=_c(s), alpha=alpha,
                 label=f'{s}  med={med:.2f}  f={f_s}')
    ax_cum.plot( bc, cum, lw=2.0, color=_c(s), alpha=alpha,
                 label=f'{s}  med={med:.2f}')

if csv_fig13 is not None:
    bc_csv = csv_fig13['bin_center'].values
    surv_cols = [c for c in csv_fig13.columns if c != 'bin_center']
    for s in sorted(surv_cols):
        h = csv_fig13[s].values
        if not np.any(np.isfinite(h)): continue
        cum = np.cumsum(np.where(np.isfinite(h),h,0)) * (bc_csv[1]-bc_csv[0])
        nf  = np.nan
        if csv_fig13s is not None:
            row = csv_fig13s[csv_fig13s['Survey']==s]
            if len(row): nf = row['Norm_factor'].values[0]
        med_ = np.nan
        if csv_fig13s is not None:
            row = csv_fig13s[csv_fig13s['Survey']==s]
            if len(row): med_ = row['Median_err_km_s'].values[0]
        f_s  = f'{nf:.2f}' if np.isfinite(nf) else 'n/a'
        m_s  = f'{med_:.2f}' if np.isfinite(med_) else 'n/a'
        ax_h.step(bc_csv, h,   where='mid', lw=2.0, color=_c(s), alpha=ALPHA,
                  label=f'{s}  med={m_s}  f={f_s}')
        ax_c.plot( bc_csv, cum, lw=2.0, color=_c(s), alpha=ALPHA,
                   label=f'{s}  med={m_s}')
elif nerr is not None:
    for s, nd in sorted(nerr.items()):
        ea  = nd['errors']; f   = nd.get('factor', np.nan)
        _add_fig13_survey(s, ea, f, ax_h, ax_c)

note = ("Fig. 13 = pipeline σ × DUP/TCH factor\n"
        "NOT affected by Bevington calibration.\n"
        "Bevington δΔRV enters Fig. 17 only.")
ax_h.text(0.97, 0.97, note, transform=ax_h.transAxes,
          fontsize=8, va='top', ha='right',
          bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', alpha=0.85))

for ax, title in zip([ax_h, ax_c], ['Density', 'Cumulative fraction']):
    ax.set_xlim(*X_LIM); ax.set_xlabel(X_LABEL, fontsize=12)
    ax.set_title(title, fontsize=12); ax.legend(fontsize=8); ax.grid(True)

ax_h.set_ylabel('Density', fontsize=12)
ax_c.set_ylabel('Cumulative fraction', fontsize=12)
ax_c.axhline(0.5, color='gray', ls=':', lw=1); ax_c.set_ylim(0,1)
add_panel_label(ax_h, '(a)'); add_panel_label(ax_c, '(b)')
fig.suptitle('Normalised RV Error Distributions  (Fig. 13)\n'
             'Per-survey pipeline errors × DUP/TCH normalization factor',
             fontsize=12, y=1.01)
fig.tight_layout()
savefig(fig, 'homo_fig13_norm_errors')
plt.show()


## Fig 15 — δRV and σRV vs stellar parameters (merged catalogue)

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
ERR_YLIM  = (0, 3.5)     # km/s
N_BINS    = 35
MIN_COUNT = 10
FIG_SIZE  = (16, 11)
# ─────────────────────────────────────────────────────────────────────────────

if csv_fig17 is None:
    print("fig17_drv_srv.csv not found — Fig 16 requires merged catalogue data.")
else:
    # NOTE: fig17_drv_srv.csv has delta_rv and sigma_rv but NOT stellar params.
    # For the vs-parameter plots we need the phase8 nerr (which has per-survey
    # median errors) or phase7 diag dataframes.
    # We plot δRV and σRV global distributions split by survey using scal diag.

    params16 = ['Teff','logg','FeH','Gmag']
    param_lbl16 = {
        'Teff': r'$T_{\rm eff}$ (K)',
        'logg': r'$\log g$ (dex)',
        'FeH' : '[Fe/H] (dex)',
        'Gmag': 'G magnitude (mag)',
    }

    # Collect drv from scal diag (Eq8 residuals ≈ σ after calibration)
    # and nerr for the normalized errors.
    if scal is not None:
        fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE)
        axes = axes.flatten()
        panel_lbl = ['(a)','(b)','(c)','(d)']
        for pi, pname in enumerate(params16):
            ax = axes[pi]
            for surv, sres in sorted(scal.items()):
                if not isinstance(sres, dict): continue
                if 'diag' not in sres: continue
                df = sres['diag']
                if pname not in df.columns or df[pname].isna().all(): continue
                x = df[pname].values.astype(float)
                # |drv| as proxy for per-star calibration residual
                y = np.abs(df['drv'].values.astype(float))
                bc, meds, _ = bin_stat(x, y, n_bins=N_BINS, min_count=MIN_COUNT)
                if len(bc)==0: continue
                ax.plot(bc, meds, '-', lw=1.8, color=_c(surv), alpha=0.85, label=surv)
            ax.set_xlabel(param_lbl16[pname], fontsize=12)
            ax.set_ylabel(r'Median $|\Delta$RV$|$  (km/s)', fontsize=11)
            ax.set_title(f'Calibration residual vs {pname}', fontsize=12)
            ax.set_ylim(*ERR_YLIM)
            ax.legend(fontsize=8, loc='upper right'); ax.grid(True)
            add_panel_label(ax, panel_lbl[pi])
        fig.suptitle('RV Calibration Residuals vs Stellar Parameters  (Fig. 16 proxy)',
                     fontsize=13, y=1.01)
        fig.tight_layout()
        savefig(fig, 'homo_fig16_drv_params')
        plt.show()
    else:
        print("Phase-7 checkpoint not found — cannot plot Fig 16 vs parameters.")


## Fig 16 — Distribution of δRV and σRV after homogenisation

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
X_LIM_FULL = (0, 8)      # Panel 0: full range (km/s)
X_LIM_ZOOM = (0, 3.5)    # Panel 1: zoomed range (km/s)
BINS_FULL  = np.linspace(0, 8,  161)
BINS_ZOOM  = np.linspace(0, 3.5, 161)
ALPHA      = 0.88
FIG_SIZE   = (22, 6.5)
# ─────────────────────────────────────────────────────────────────────────────

if csv_fig17 is None:
    print("fig17_drv_srv.csv not found.")
else:
    drv_all = csv_fig17['delta_rv'].dropna().values
    srv_all = csv_fig17['sigma_rv'].dropna().values
    drv_all = drv_all[drv_all > 0]; srv_all = srv_all[srv_all > 0]

    # Stats from CSV summary
    stats = csv_fig17s.iloc[0].to_dict() if csv_fig17s is not None else {}

    def _mode(arr, bins):
        h, e = np.histogram(arr[arr<=e[-1]] if len(arr) else arr, bins=bins, density=True)
        bc_  = 0.5*(e[:-1]+e[1:])
        return float(bc_[np.argmax(h)]) if len(h) else float(np.median(arr))

    drv_mode = _mode(drv_all[drv_all<=X_LIM_ZOOM[1]], BINS_ZOOM)
    srv_mode = _mode(srv_all[srv_all<=X_LIM_ZOOM[1]], BINS_ZOOM) if len(srv_all)>0 else np.nan
    drv_med  = stats.get('drv_median_km_s', float(np.median(drv_all)))
    srv_med  = stats.get('srv_median_km_s', float(np.median(srv_all)) if len(srv_all)>0 else np.nan)
    drv_p5   = stats.get('drv_p5_km_s',  float(np.percentile(drv_all,5)))
    drv_p95  = stats.get('drv_p95_km_s', float(np.percentile(drv_all,95)))
    srv_p5   = stats.get('srv_p5_km_s',  float(np.percentile(srv_all,5))  if len(srv_all)>0 else np.nan)
    srv_p95  = stats.get('srv_p95_km_s', float(np.percentile(srv_all,95)) if len(srv_all)>0 else np.nan)

    fig, axes = plt.subplots(1, 3, figsize=FIG_SIZE)
    ax0, ax1, ax2 = axes

    # ── Panel 0: Full range ───────────────────────────────────────────────────
    bc_f = 0.5*(BINS_FULL[:-1]+BINS_FULL[1:])
    h_d, _ = np.histogram(drv_all, bins=BINS_FULL, density=True)
    ax0.step(bc_f, h_d, where='mid', lw=2.5, color='tomato',
             label=f'δRV  (N={len(drv_all):,})')
    if len(srv_all)>0:
        h_s, _ = np.histogram(srv_all, bins=BINS_FULL, density=True)
        ax0.step(bc_f, h_s, where='mid', lw=2.5, color='steelblue',
                 label=f'σRV  (N={len(srv_all):,})')
    ax0.axvline(drv_mode, color='tomato',   ls='--', lw=1.5, alpha=0.75,
                label=f'δRV mode = {drv_mode:.3f}')
    if np.isfinite(srv_mode):
        ax0.axvline(srv_mode, color='steelblue', ls='--', lw=1.5, alpha=0.75,
                    label=f'σRV mode = {srv_mode:.3f}')
    stat_txt = (f"δRV  med={drv_med:.3f}  [{drv_p5:.3f}, {drv_p95:.3f}] km/s\n"
                f"σRV  med={srv_med:.3f}  [{srv_p5:.3f}, {srv_p95:.3f}] km/s"
                if np.isfinite(srv_med) else
                f"δRV  med={drv_med:.3f}  [{drv_p5:.3f}, {drv_p95:.3f}] km/s")
    ax0.text(0.97,0.97, stat_txt, transform=ax0.transAxes, fontsize=8.5,
             va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', alpha=0.9))
    ax0.set_xlim(*X_LIM_FULL); ax0.set_xlabel('Error (km/s)', fontsize=12)
    ax0.set_ylabel('Density', fontsize=12); ax0.set_title('Full range', fontsize=12)
    ax0.legend(fontsize=9); ax0.grid(True)
    add_panel_label(ax0, '(a)')

    # ── Panel 1: Zoomed with fill ─────────────────────────────────────────────
    bc_z = 0.5*(BINS_ZOOM[:-1]+BINS_ZOOM[1:])
    drv_z = drv_all[drv_all <= X_LIM_ZOOM[1]]
    srv_z = srv_all[srv_all <= X_LIM_ZOOM[1]] if len(srv_all)>0 else np.array([])
    h_dz, _ = np.histogram(drv_z, bins=BINS_ZOOM, density=True)
    ax1.fill_between(bc_z, h_dz, step='mid', alpha=0.20, color='tomato')
    ax1.step(bc_z, h_dz, where='mid', lw=2.5, color='tomato',
             label=f'δRV  med={drv_med:.3f} km/s')
    if len(srv_z)>0:
        h_sz, _ = np.histogram(srv_z, bins=BINS_ZOOM, density=True)
        ax1.fill_between(bc_z, h_sz, step='mid', alpha=0.20, color='steelblue')
        ax1.step(bc_z, h_sz, where='mid', lw=2.5, color='steelblue',
                 label=f'σRV  med={srv_med:.3f} km/s')
    ax1.axvline(drv_med, color='tomato',   ls=':', lw=2.0, alpha=0.8)
    if np.isfinite(srv_med): ax1.axvline(srv_med, color='steelblue', ls=':', lw=2.0, alpha=0.8)
    note_bev = ("Gaia δRV includes Bevington\npolynomial uncertainty:\n"
                r"δRV$_{\rm calib}$ = $\sqrt{δΔRV^2 + δRV_{\rm Gaia}^2}$")
    ax1.text(0.97,0.97, note_bev, transform=ax1.transAxes,
             fontsize=8, va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.4', fc='#e8f4f8', alpha=0.92))
    ax1.set_xlim(*X_LIM_ZOOM); ax1.set_xlabel('Error (km/s)', fontsize=12)
    ax1.set_ylabel('Density', fontsize=12); ax1.set_title('Zoomed view', fontsize=12)
    ax1.legend(fontsize=9); ax1.grid(True)
    add_panel_label(ax1, '(b)')

    # ── Panel 2: Per-survey breakdown from nerr ────────────────────────────────
    if nerr is not None:
        for s in sorted(nerr.keys()):
            ea = nerr[s]['errors']
            ea = ea[(ea>0) & (ea<=X_LIM_ZOOM[1]) & np.isfinite(ea)]
            if len(ea)<20: continue
            h_s, _ = np.histogram(ea, bins=BINS_ZOOM, density=True)
            med_s  = float(np.median(ea))
            ax2.step(bc_z, h_s, where='mid', lw=1.9, color=_c(s), alpha=ALPHA,
                     label=f'{s}  med={med_s:.3f}  N={len(ea):,}')
    ax2.set_xlim(*X_LIM_ZOOM); ax2.set_xlabel(r'$\delta$RV  (km/s)', fontsize=12)
    ax2.set_ylabel('Density', fontsize=12)
    ax2.set_title('Per-survey normalised errors', fontsize=11)
    ax2.legend(fontsize=8, loc='upper right'); ax2.grid(True)
    add_panel_label(ax2, '(c)')

    fig.suptitle(
        r'Distribution of $\delta$RV and $\sigma$RV after Homogenisation  (Fig. 17)' + '\n'
        r'$\delta$RV = weighted-mean uncertainty  |  $\sigma$RV = inter-survey scatter (Eq. 9)',
        fontsize=12, y=1.02)
    fig.tight_layout()
    savefig(fig, 'homo_fig17_drv_srv')
    plt.show()


## Table 4 — Combined normalisation factors

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
FIG_SIZE  = (11, 4.5)
BAR_WIDTH = 0.28
# ─────────────────────────────────────────────────────────────────────────────

if csv_nf is not None:
    display(csv_nf.style.format({'Combined_Factor':'{:.3f}','DUP_factor':'{:.3f}',
                                  'TCH_factor':'{:.3f}'}).set_caption('Table 4: Normalisation Factors'))

    surveys = csv_nf['Survey'].values
    x = np.arange(len(surveys))

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    for i,(col,lbl,clr) in enumerate([
            ('DUP_factor','DUP','#5aae61'),
            ('TCH_factor','TCH','#4393c3'),
            ('Combined_Factor','Combined','#d6604d')]):
        vals = pd.to_numeric(csv_nf[col], errors='coerce').values
        ax.bar(x + (i-1)*BAR_WIDTH, vals, width=BAR_WIDTH, label=lbl,
               color=clr, alpha=0.85, edgecolor='white', lw=0.8)
    ax.axhline(1.0, color='k', ls='--', lw=1.3, alpha=0.5, label='f = 1.0')
    ax.set_xticks(x); ax.set_xticklabels(surveys, fontsize=11)
    ax.set_ylabel('Normalisation factor $f$', fontsize=12)
    ax.set_title('Combined RV error normalisation factors  (Table 4)', fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, axis='y')
    add_panel_label(ax, ' ')
    fig.tight_layout()
    savefig(fig, 'homo_table4_norm_factors')
    plt.show()


## Zero-point shifts (Gaia calibration)

In [ ]:
if csv_zp is not None and len(csv_zp) > 0:
    pivot = csv_zp.pivot(index='Survey', columns='Equation', values='ZP_shift_km_s')
    display(pivot.style.format('{:.3f}').set_caption('Zero-point shifts ZP (km/s) per equation per survey'))

    fig, ax = plt.subplots(figsize=(10, 4.5))
    eqs = sorted(csv_zp['Equation'].unique())
    survs = sorted(csv_zp['Survey'].unique())
    x = np.arange(len(survs))
    w = 0.8 / len(eqs)
    for i, eq in enumerate(eqs):
        sub = csv_zp[csv_zp['Equation']==eq].set_index('Survey')
        vals = [sub.loc[s,'ZP_shift_km_s'] if s in sub.index else np.nan for s in survs]
        ax.bar(x + (i - len(eqs)/2 + 0.5)*w, vals, width=w, label=eq, alpha=0.85,
               edgecolor='white', lw=0.8)
    ax.axhline(0, color='k', ls='--', lw=1); ax.set_xticks(x)
    ax.set_xticklabels(survs, fontsize=11)
    ax.set_ylabel('ZP shift  (km/s)', fontsize=12)
    ax.set_title(r'Gaia calibration ZP shifts: $\langle\Delta$RV$\rangle$ per survey per equation',
                 fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, axis='y')
    fig.tight_layout()
    savefig(fig, 'homo_gaia_zp_shifts')
    plt.show()


## Supplementary — DUP raw ΔRV distributions (km/s)

In [ ]:
# ── Adjustable parameters ─────────────────────────────────────────────────────
X_LIM   = (-15, 15)   # km/s
N_BINS  = 120
ALPHA   = 0.80
# ─────────────────────────────────────────────────────────────────────────────

if dup is not None:
    survs_dup = sorted(dup.keys())
    n = len(survs_dup)
    ncols = min(3, n); nrows = (n+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.5*nrows),
                             squeeze=False)
    bins = np.linspace(*X_LIM, N_BINS+1); bc = 0.5*(bins[:-1]+bins[1:])
    for si, s in enumerate(survs_dup):
        ax = axes[si//ncols][si%ncols]
        rd = dup[s]['raw_diffs']
        rd = rd[(rd>=X_LIM[0]) & (rd<=X_LIM[1])]
        if len(rd)>0:
            h, _ = np.histogram(rd, bins=bins, density=True)
            ax.step(bc, h, where='mid', lw=2.0, color=_c(s), alpha=ALPHA)
            ax.fill_between(bc, h, step='mid', color=_c(s), alpha=0.15)
            med = float(np.median(rd)); mad = float(np.median(np.abs(rd-med)))
            ax.axvline(0,  color='k',   ls=':', lw=1)
            ax.axvline(med, color='r',  ls='--', lw=1.5, label=f'med={med:+.3f}')
            ax.text(0.97,0.97, f'N={len(rd):,}\nMAD={mad:.3f} km/s',
                    transform=ax.transAxes, fontsize=9, va='top', ha='right',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))
        ax.set_xlim(*X_LIM); ax.set_xlabel(r'$\Delta$RV  (km/s)', fontsize=11)
        ax.set_ylabel('Density', fontsize=10); ax.set_title(s, fontsize=12)
        ax.legend(fontsize=9); ax.grid(True)
    # Hide unused panels
    for si in range(n, nrows*ncols): axes[si//ncols][si%ncols].set_visible(False)
    fig.suptitle('DUP Method: Raw ΔRV distributions per survey', fontsize=13, y=1.01)
    fig.tight_layout()
    savefig(fig, 'homo_supp_dup_raw_drv')
    plt.show()


## Supplementary — TCH pairwise σ table

In [ ]:
if tch is not None:
    pw = tch.get('pairwise', {})
    cf = tch.get('combined_factors', {})
    rows = []
    for (si,sj), d in sorted(pw.items()):
        rows.append({'Survey_1':si,'Survey_2':sj,
                     'sigma_km_s':d['sigma'],'N_common':d['n']})
    df_pw = pd.DataFrame(rows).sort_values('sigma_km_s')
    display(df_pw.style.format({'sigma_km_s':'{:.4f}'}).set_caption('TCH pairwise σ'))

    if cf:
        df_cf = pd.DataFrame([{'Survey':s,'Combined_factor':v} for s,v in cf.items()])
        display(df_cf.style.format({'Combined_factor':'{:.4f}'}).set_caption('Combined normalisation factors'))


---
## All Figures Saved

In [ ]:
import glob as _glob
saved = sorted(_glob.glob(os.path.join(OUT_DIR, '*.png')))
print(f'\n{len(saved)} PNG files in {OUT_DIR}:')
for p in saved:
    print(f'  {os.path.basename(p)}')
